# LeetCode 291 — Word Pattern II bằng Python

> **Lưu ý: đây là bài Premium** (bài bị khóa trên LeetCode, chỉ tài khoản trả phí mới mở được). Phát biểu đề, ví dụ và constraints dưới đây được viết lại theo bản LeetCode mà mình biết.
> Danh mục học của bạn xếp bài này vào nhóm **Hard**; theo những gì mình biết, LeetCode hiện gắn nhãn **Medium** (trước đây từng là Hard). Dù nhãn là gì, đây là bài rất hay để học **backtracking (quay lui)** trên chuỗi và kỹ thuật giữ **song ánh (bijection)** bằng `dict` + `set`.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def wordPatternMatch(self, pattern: str, s: str) -> bool:

```

## Sau notebook này, bạn sẽ nắm được

1. Đề bài yêu cầu gì: thế nào là "khớp mẫu" (match) và vì sao phải là **song ánh** giữa ký tự của `pattern` và **chuỗi con khác rỗng** của `s`.
2. Ôn lại bài dễ hơn **#290 Word Pattern** (có dấu cách) và học cách kiểm tra song ánh bằng `dict` + `set` — kèm **phản ví dụ** cho thấy chỉ kiểm tra **một chiều** là sai.
3. Bốn cách giải, đi từ trực giác đến khéo léo:
   - **Cách 1:** Brute force — sinh **mọi cách cắt** `s` thành `len(pattern)` mảnh bằng `itertools.combinations`, rồi kiểm tra song ánh. Chậm nhưng chắc chắn đúng → dùng làm **trọng tài** khi test.
   - **Cách 2 (cách chính):** **Backtracking** — gán dần từng ký tự: ký tự đã có ánh xạ thì chỉ cần **kiểm tra** `s.startswith(...)`; ký tự mới thì **thử** mọi tiền tố chưa bị dùng, gán rồi **gỡ**.
   - **Cách 3:** Backtracking + **cắt tỉa theo độ dài** — biết "phần còn lại cần ít nhất bao nhiêu ký tự" để bỏ sớm những nhánh chắc chắn hỏng.
   - **Cách 4 (tùy chọn):** **Đoán độ dài trước** cho mỗi ký tự khác nhau (giải phương trình `Σ count·len = n`) rồi mới kiểm tra.
4. Kiến thức nền dạy từ đầu: slicing, `str.startswith`, `str.count`, `dict`/`set` (thêm, tra, gỡ), `itertools.combinations`, `itertools.product`, `collections.Counter`, **đệ quy** và khung **chọn → đi tiếp → gỡ** của backtracking.
5. Trace (mô phỏng) cây gọi đệ quy cho `"abab"` / `"redblueredblue"`, code nộp LeetCode, test đối chiếu chéo với brute force, lỗi thường gặp, bài tập + đáp án.

## 1. Đề bài

Cho một chuỗi mẫu `pattern` và một chuỗi `s`, trả về `true` nếu `s` **khớp** (match) với `pattern`.

Chuỗi `s` **khớp** với `pattern` nếu tồn tại một **ánh xạ song ánh** (bijective mapping) từ **các ký tự đơn** (của `pattern`) sang **các chuỗi khác rỗng** sao cho: khi thay mỗi ký tự trong `pattern` bằng chuỗi mà nó được ánh xạ tới, ta thu được đúng chuỗi `s`.

**Song ánh** ở đây nghĩa là: **không có hai ký tự nào ánh xạ tới cùng một chuỗi**, và **không có ký tự nào ánh xạ tới hai chuỗi khác nhau**.

**Ví dụ 1**

```text
Input:  pattern = "abab", s = "redblueredblue"
Output: true
Explanation: One possible mapping is as follows:
'a' -> "red"
'b' -> "blue"
```

**Ví dụ 2**

```text
Input:  pattern = "aaaa", s = "asdasdasdasd"
Output: true
Explanation: One possible mapping is as follows:
'a' -> "asd"
```

**Ví dụ 3**

```text
Input:  pattern = "aabb", s = "xyzabcxzyabc"
Output: false
```

**Constraints (ràng buộc):**

- `1 <= pattern.length, s.length <= 20`
- `pattern` và `s` chỉ gồm chữ cái tiếng Anh viết thường.

Hai điều cần để ý ngay từ đầu:

- Đề chỉ hỏi **có tồn tại** một ánh xạ hay không (trả `True/False`), **không** bắt in ra ánh xạ. Chữ "One possible mapping" (một ánh xạ có thể) gợi ý rằng có thể có **nhiều** ánh xạ hợp lệ — ta sẽ thấy Ví dụ 1 có tới 6 ánh xạ!
- `s` dài tối đa **20** — rất nhỏ. Đây là "tín hiệu" rằng đề chấp nhận lời giải **thử nhiều khả năng** (hàm mũ) miễn là thử khéo.

In [1]:
examples = [
    ("abab", "redblueredblue", True),
    ("aaaa", "asdasdasdasd", True),
    ("aabb", "xyzabcxzyabc", False),
]

for pattern, s, expected in examples:
    print(f"pattern = {pattern!r:8} s = {s!r:18} len(pattern) = {len(pattern)}, len(s) = {len(s):2}  ->  expected = {expected}")

pattern = 'abab'   s = 'redblueredblue'   len(pattern) = 4, len(s) = 14  ->  expected = True
pattern = 'aaaa'   s = 'asdasdasdasd'     len(pattern) = 4, len(s) = 12  ->  expected = True
pattern = 'aabb'   s = 'xyzabcxzyabc'     len(pattern) = 4, len(s) = 12  ->  expected = False


### Hiểu đề bằng tay: "thay ký tự bằng chuỗi, rồi nối lại"

Với Ví dụ 1, ánh xạ `a -> "red"`, `b -> "blue"`. Thay từng ký tự của `pattern = "abab"`:

```text
pattern:   a      b       a      b
thay bằng: "red"  "blue"  "red"  "blue"
nối lại:   "red" + "blue" + "red" + "blue" = "redblueredblue"  == s  ✓
```

Một ánh xạ được coi là **hợp lệ** khi thỏa **đủ 4 điều**:

1. **Mọi** ký tự trong `pattern` đều có chuỗi để thay.
2. Mỗi chuỗi đó **khác rỗng** (không được ánh xạ `a -> ""`).
3. Hai ký tự **khác nhau** phải ánh xạ tới hai chuỗi **khác nhau** (điều kiện "song ánh").
4. Nối các chuỗi theo đúng thứ tự `pattern` ra **đúng bằng** `s` (không thừa, không thiếu).

Điều 3 là chỗ dễ quên nhất. Ví dụ `pattern = "ab"`, `s = "xx"`: nếu cho `a -> "x"` và `b -> "x"` thì nối lại đúng là `"xx"`, nhưng **vi phạm** điều 3 (hai ký tự khác nhau dùng chung chuỗi `"x"`) → không hợp lệ. Thử hết các cách khác (`a -> "xx"` thì `b` không còn gì để nhận) → đáp án là `False`.

Cell dưới viết 2 hàm nhỏ để kiểm tra bằng code: `apply_mapping` (thay rồi nối) và `is_valid_mapping` (kiểm tra đủ 4 điều).

In [2]:
def apply_mapping(pattern: str, mapping: dict) -> str:
    # Thay từng ký tự của pattern bằng chuỗi của nó, rồi nối lại
    return "".join(mapping[ch] for ch in pattern)


def is_valid_mapping(pattern: str, s: str, mapping: dict) -> bool:
    if any(ch not in mapping for ch in pattern):          # điều 1
        return False
    if any(v == "" for v in mapping.values()):            # điều 2
        return False
    if len(set(mapping.values())) != len(mapping):        # điều 3: các chuỗi phải khác nhau
        return False
    return apply_mapping(pattern, mapping) == s           # điều 4


print(apply_mapping("abab", {"a": "red", "b": "blue"}))
print(is_valid_mapping("abab", "redblueredblue", {"a": "red", "b": "blue"}))   # True
print(is_valid_mapping("aaaa", "asdasdasdasd", {"a": "asd"}))                  # True
print(is_valid_mapping("ab", "xx", {"a": "x", "b": "x"}))                      # False: vi phạm điều 3
print(is_valid_mapping("ab", "xx", {"a": "xx", "b": ""}))                      # False: vi phạm điều 2

redblueredblue
True
True
False
False


## 2. Kiến thức nền Python cho bài này

### 2.1. Cắt chuỗi (slicing), `startswith` và `count`

- `s[j:end]` lấy các ký tự từ chỉ số `j` **đến trước** `end` (không lấy `end`). Độ dài mảnh là `end - j`.
- Với `s = "redblueredblue"` (chỉ số `0..13`): `s[0:3] = "red"`, `s[3:7] = "blue"`, `s[7:]` là phần từ chỉ số 7 tới hết.

| Biểu thức | Kết quả | Ý nghĩa trong bài |
|---|---|---|
| `s[0:3]` | `"red"` | một **ứng viên** (candidate) cho ký tự đầu |
| `s[3:7]` | `"blue"` | mảnh tiếp theo |
| `s.startswith("red", 7)` | `True` | "từ vị trí 7, `s` có bắt đầu bằng `"red"` không?" |
| `s.startswith("red", 3)` | `False` | ở vị trí 3 là `"blue..."` |
| `"abab".count("a", 1)` | `1` | đếm `'a'` trong `"abab"` **tính từ chỉ số 1** (`"bab"`) |

**`s.startswith(word, j)`** là công cụ quan trọng nhất của Cách 2: khi một ký tự **đã** có ánh xạ `word`, ta chỉ cần hỏi "tại vị trí `j`, `s` có bắt đầu bằng `word` không?". Nó tương đương `s[j:j + len(word)] == word` nhưng không cần tự tính chỉ số cuối (và không tạo chuỗi con mới).

**`pattern.count(ch, i)`** đếm số lần ký tự `ch` xuất hiện trong `pattern` **từ chỉ số `i` trở đi** — ta sẽ dùng ở Cách 3 để biết "ký tự này còn phải xuất hiện mấy lần nữa".

In [3]:
s = "redblueredblue"
print(s[0:3], s[3:7], s[7:])          # red blue redblue
print(len(s[3:7]), 7 - 3)              # độ dài mảnh = end - j

print(s.startswith("red", 7))          # True
print(s.startswith("red", 3))          # False
print(s[7:7 + len("red")] == "red")    # cách viết dài tương đương

# Mọi tiền tố (prefix) của s[j:] — chính là các "ứng viên" ta sẽ thử
j = 7
print([s[j:end] for end in range(j + 1, j + 5)])   # 4 ứng viên đầu tiên

print("abab".count("a"), "abab".count("a", 1), "abab".count("a", 3))   # 2 1 0

red blue redblue
4 4
True
False
True
['r', 're', 'red', 'redb']
2 1 0


### 2.2. `dict` lưu ánh xạ, `set` lưu các chuỗi "đã có chủ"

- **`dict`** (từ điển) lưu cặp `key -> value`. Ở đây: `mapping[ch] = word` nghĩa là "ký tự `ch` được thay bằng chuỗi `word`".
  - Thêm/gán: `mapping["a"] = "red"`.
  - Kiểm tra có chưa: `"a" in mapping` (trung bình `O(1)`).
  - Đọc ra: `mapping["a"]`.
  - Gỡ đi: `del mapping["a"]`.
- **`set`** (tập hợp) lưu các phần tử **không trùng nhau**. Ở đây: `used` chứa các chuỗi **đã được một ký tự nào đó nhận**.
  - Thêm: `used.add("red")`; kiểm tra: `"red" in used` (trung bình `O(1)`); gỡ: `used.remove("red")`.

**Vì sao cần thêm `set` khi đã có `dict`?** `dict` chỉ tra nhanh theo **key** (ký tự). Câu hỏi "chuỗi `"red"` đã có ký tự nào nhận chưa?" là câu hỏi theo **value**. Ta có thể viết `"red" in mapping.values()`, nhưng như vậy phải duyệt hết các value (`O(k)` với `k` = số ký tự đã gán). `set used` trả lời câu đó trong `O(1)` trung bình — và quan trọng hơn, nó làm **chiều thứ hai** của song ánh trở nên rõ ràng trong code.

Trong backtracking, `dict` và `set` luôn đi **cặp**: gán thì thêm vào cả hai, gỡ thì xóa khỏi cả hai. Cell dưới mô phỏng đúng thao tác "gán rồi gỡ" đó.

In [4]:
mapping = {}
used = set()

# Gán a -> "red"
mapping["a"] = "red"
used.add("red")
print("sau khi gán :", mapping, used)

print("'a' đã có ánh xạ?", "a" in mapping, "->", mapping["a"])
print("'b' đã có ánh xạ?", "b" in mapping)
print("'red' đã có chủ? ", "red" in used)
print("'blue' đã có chủ?", "blue" in used)

# Gỡ a -> "red" (quay lui)
del mapping["a"]
used.remove("red")
print("sau khi gỡ  :", mapping, used)

sau khi gán : {'a': 'red'} {'red'}
'a' đã có ánh xạ? True -> red
'b' đã có ánh xạ? False
'red' đã có chủ?  True
'blue' đã có chủ? False
sau khi gỡ  : {} set()


### 2.3. Song ánh (bijection) là gì? — hai chiều phải cùng đúng

Hình dung hai cột: bên trái là các **ký tự** của `pattern`, bên phải là các **chuỗi**. Mỗi ký tự được nối với chuỗi của nó bằng một mũi tên.

```text
  ĐÚNG (song ánh)          SAI kiểu 1                  SAI kiểu 2
  a ──> "red"              a ──> "red"                 a ──> "dog"
  b ──> "blue"             a ──> "blue"   (a có 2      b ──> "dog"   (2 ký tự chung
                                           chuỗi!)                     1 chuỗi!)
```

- **Chiều 1 — ký tự → chuỗi phải nhất quán:** mỗi ký tự chỉ có **một** chuỗi. Gặp lại `a` ở chỗ khác thì phải thấy lại đúng `"red"`. (Trong toán học: đây là điều kiện để nó là một **hàm**.)
- **Chiều 2 — chuỗi → ký tự phải nhất quán:** mỗi chuỗi chỉ thuộc về **một** ký tự. Hai ký tự khác nhau không được dùng chung chuỗi. (Trong toán học: **đơn ánh** — injective.)

Khi cả hai chiều đúng, mũi tên là "một-một": nhìn ký tự biết chuỗi, nhìn chuỗi biết ký tự → **song ánh** (giữa tập ký tự xuất hiện trong `pattern` và tập chuỗi được dùng).

**Bẫy của `dict`:** nếu ta "nạp" lần lượt các cặp `(ký tự, chuỗi)` vào `dict` bằng phép gán `d[ch] = w`, vi phạm chiều 1 sẽ bị **ghi đè im lặng** (không báo lỗi!), còn vi phạm chiều 2 thì `dict` hoàn toàn không để ý. Cell dưới cho thấy cả hai.

In [5]:
# SAI kiểu 1: 'a' gặp hai chuỗi khác nhau -> dict ghi đè im lặng
pairs1 = [("a", "red"), ("b", "blue"), ("a", "blue")]
d1 = {}
for ch, w in pairs1:
    d1[ch] = w
print("Kiểu 1 sau khi nạp:", d1, " <- mất dấu vết 'a' -> 'red', không có lỗi nào được báo!")

# SAI kiểu 2: hai ký tự dùng chung một chuỗi -> dict không phát hiện
pairs2 = [("a", "dog"), ("b", "dog")]
d2 = {}
for ch, w in pairs2:
    d2[ch] = w
print("Kiểu 2 sau khi nạp:", d2)
print("Số ký tự:", len(d2), "| số chuỗi khác nhau:", len(set(d2.values())), " <- lệch nhau => không đơn ánh")

Kiểu 1 sau khi nạp: {'a': 'blue', 'b': 'blue'}  <- mất dấu vết 'a' -> 'red', không có lỗi nào được báo!
Kiểu 2 sau khi nạp: {'a': 'dog', 'b': 'dog'}
Số ký tự: 2 | số chuỗi khác nhau: 1  <- lệch nhau => không đơn ánh


## 3. Ôn bài dễ hơn: #290 Word Pattern (có dấu cách)

Bài **#290 Word Pattern** (Easy) gần như giống hệt, chỉ khác một điều: `s` có **dấu cách** ngăn giữa các từ, nên ta **biết sẵn** cách cắt.

> Cho `pattern` và chuỗi `s`, kiểm tra `s` có **theo** (follow) cùng mẫu hay không: phải có song ánh giữa mỗi chữ cái trong `pattern` và mỗi **từ** khác rỗng trong `s`.
>
> - `pattern = "abba"`, `s = "dog cat cat dog"` → `true`
> - `pattern = "abba"`, `s = "dog cat cat fish"` → `false`
> - `pattern = "aaaa"`, `s = "dog cat cat dog"` → `false`
>
> Constraints: `1 <= pattern.length <= 300`; `pattern` chỉ gồm chữ thường; `1 <= s.length <= 3000`; `s` gồm chữ thường và dấu cách `' '`; không có dấu cách ở đầu/cuối; các từ cách nhau đúng **một** dấu cách.

Hai công cụ Python mới:

- `s.split()` tách chuỗi theo dấu cách → list các từ.
- `zip(pattern, words)` ghép cặp phần tử thứ `t` của hai dãy: `('a','dog'), ('b','cat'), ...`. Lưu ý: `zip` **dừng ở dãy ngắn hơn**, nên phải tự kiểm tra `len(words) == len(pattern)` trước.

In [6]:
s290 = "dog cat cat dog"
words = s290.split()
print(words, len(words))

for ch, w in zip("abba", words):
    print(ch, "->", w)

print(list(zip("abc", ["x", "y"])))   # zip dừng ở dãy ngắn hơn: mất 'c' mà không báo lỗi!

['dog', 'cat', 'cat', 'dog'] 4
a -> dog
b -> cat
b -> cat
a -> dog
[('a', 'x'), ('b', 'y')]


### 3.1. Người mới hay viết: chỉ kiểm tra **một chiều** — và vì sao sai

Ý tưởng đầu tiên thường là: "dùng một `dict` ký tự → từ; gặp ký tự đã có thì so với từ cũ". Cách này chỉ kiểm tra **chiều 1**.

**Phản ví dụ:** `pattern = "abba"`, `s = "dog dog dog dog"`.

| Bước | cặp | `dict` ký tự→từ trước | Kiểm tra chiều 1 | Hành động |
|---:|---|---|---|---|
| 0 | `a, dog` | `{}` | `a` mới | lưu `a -> dog` |
| 1 | `b, dog` | `{a: dog}` | `b` mới | lưu `b -> dog` ⚠️ (dog đã thuộc `a`!) |
| 2 | `b, dog` | `{a: dog, b: dog}` | `b` cũ, `dog == dog` | ok |
| 3 | `a, dog` | ... | `a` cũ, `dog == dog` | ok → trả `True` ❌ |

Kết quả `True` là **sai**: `a` và `b` cùng ánh xạ tới `"dog"`.

Nếu "lật lại" và chỉ kiểm tra **chiều 2** (một `dict` từ → ký tự) thì cũng sai, với phản ví dụ đối xứng: `pattern = "aaaa"`, `s = "dog cat cat dog"` (Ví dụ 3 của #290). `dict` từ → ký tự sẽ là `{dog: a, cat: a}` — không có từ nào gặp hai ký tự — nên trả `True`, trong khi đúng ra `a` đang bị ánh xạ tới **hai** từ khác nhau.

Bài học: **một chiều thôi là không đủ**, phản ví dụ luôn tồn tại theo cả hai hướng.

In [7]:
def word_pattern_one_way(pattern: str, s: str) -> bool:
    # CHỈ kiểm tra chiều 1: ký tự -> từ  (SAI)
    words = s.split()
    if len(words) != len(pattern):
        return False
    char_to_word = {}
    for ch, w in zip(pattern, words):
        if ch in char_to_word and char_to_word[ch] != w:
            return False
        char_to_word[ch] = w
    return True


def word_pattern_reverse_only(pattern: str, s: str) -> bool:
    # CHỈ kiểm tra chiều 2: từ -> ký tự  (cũng SAI)
    words = s.split()
    if len(words) != len(pattern):
        return False
    word_to_char = {}
    for ch, w in zip(pattern, words):
        if w in word_to_char and word_to_char[w] != ch:
            return False
        word_to_char[w] = ch
    return True


print("chỉ chiều 1, ('abba', 'dog dog dog dog'):", word_pattern_one_way("abba", "dog dog dog dog"), " <- đúng ra False")
print("chỉ chiều 2, ('aaaa', 'dog cat cat dog'):", word_pattern_reverse_only("aaaa", "dog cat cat dog"), " <- đúng ra False")

chỉ chiều 1, ('abba', 'dog dog dog dog'): True  <- đúng ra False
chỉ chiều 2, ('aaaa', 'dog cat cat dog'): True  <- đúng ra False


### 3.2. Đủ hai chiều: `dict` (xuôi) + `set` (các từ đã có chủ)

Cách sửa chuẩn là kiểm tra cả hai chiều. Có hai kiểu viết tương đương:

- **Hai `dict`:** `char_to_word` và `word_to_char`, kiểm tra cả hai.
- **`dict` + `set`** (gọn hơn, và **chính là kiểu ta dùng cho #291**): với mỗi cặp `(ch, w)`:
  - Nếu `ch` **đã** có ánh xạ: bắt buộc `char_to_word[ch] == w` (chiều 1).
  - Nếu `ch` **mới**: bắt buộc `w` **chưa có chủ**, tức `w not in used` (chiều 2). Rồi mới lưu `ch -> w` và `used.add(w)`.

**Vì sao `set` là đủ cho chiều 2?** Một từ `w` chỉ được thêm vào `used` đúng lúc nó được gán cho một ký tự mới. Nên "`w in used`" nghĩa là "`w` đã thuộc về một ký tự nào đó". Khi gặp lại **chính** ký tự đó, ta đi nhánh "đã có ánh xạ" (so sánh bằng nhau) chứ không đi nhánh "mới" — vì vậy không bao giờ chặn nhầm trường hợp hợp lệ.

Ta viết phần kiểm tra thành hàm `matches_pieces(pattern, pieces)` nhận **list mảnh đã cắt sẵn** — để lát nữa Cách 1 của #291 dùng lại nguyên xi. `word_pattern` của #290 chỉ là `matches_pieces(pattern, s.split())`.

In [8]:
def matches_pieces(pattern: str, pieces: list) -> bool:
    # True nếu có song ánh giữa ký tự pattern và các mảnh (đã cắt sẵn, theo đúng thứ tự)
    if len(pieces) != len(pattern):
        return False
    char_to_word = {}
    used = set()
    for ch, w in zip(pattern, pieces):
        if ch in char_to_word:
            if char_to_word[ch] != w:      # chiều 1: ký tự cũ phải gặp lại đúng từ cũ
                return False
        else:
            if w in used:                  # chiều 2: từ này đã thuộc ký tự khác
                return False
            char_to_word[ch] = w
            used.add(w)
    return True


def word_pattern(pattern: str, s: str) -> bool:
    # LeetCode #290
    return matches_pieces(pattern, s.split())


tests_290 = [
    ("abba", "dog cat cat dog", True),
    ("abba", "dog cat cat fish", False),
    ("aaaa", "dog cat cat dog", False),
    ("abba", "dog dog dog dog", False),   # phản ví dụ của "chỉ chiều 1"
    ("aaa", "dog dog", False),            # lệch độ dài
]
for p, s, expected in tests_290:
    got = word_pattern(p, s)
    print(f"{p!r:7} {s!r:20} -> {got!s:5} (expected {expected})")
    assert got == expected
print("#290: PASS cả 5 test")

'abba'  'dog cat cat dog'    -> True  (expected True)
'abba'  'dog cat cat fish'   -> False (expected False)
'aaaa'  'dog cat cat dog'    -> False (expected False)
'abba'  'dog dog dog dog'    -> False (expected False)
'aaa'   'dog dog'            -> False (expected False)
#290: PASS cả 5 test


### 3.3. Trace #290: nhìn `dict` và `set` thay đổi

Trace `pattern = "abba"`, `s = "dog cat cat fish"`:

| t | `ch` | `w` | `char_to_word` trước | `used` trước | Nhánh | Hành động |
|---:|---|---|---|---|---|---|
| 0 | a | dog | `{}` | `{}` | mới, `dog` chưa có chủ | lưu `a -> dog` |
| 1 | b | cat | `{a: dog}` | `{dog}` | mới, `cat` chưa có chủ | lưu `b -> cat` |
| 2 | b | cat | `{a: dog, b: cat}` | `{dog, cat}` | cũ, `cat == cat` | ok |
| 3 | a | fish | `{a: dog, b: cat}` | `{dog, cat}` | cũ, `dog != fish` | trả `False` |

Và với phản ví dụ `"abba"`, `"dog dog dog dog"`: ở `t = 1`, `b` là ký tự mới nhưng `dog` **đã có chủ** → `False` ngay. Đó chính là dòng mà bản "một chiều" bỏ sót.

In [9]:
def trace_word_pattern(pattern: str, s: str) -> bool:
    pieces = s.split()
    print(f"pattern={pattern!r}, pieces={pieces}")
    char_to_word, used = {}, set()
    for t, (ch, w) in enumerate(zip(pattern, pieces)):
        before = f"map={char_to_word}, used={sorted(used)}"
        if ch in char_to_word:
            ok = char_to_word[ch] == w
            print(f"  t={t} {ch}->{w!r:6} | {before} | cũ: {char_to_word[ch]!r} == {w!r}? {ok}")
            if not ok:
                return False
        else:
            if w in used:
                print(f"  t={t} {ch}->{w!r:6} | {before} | mới nhưng {w!r} đã có chủ -> False")
                return False
            char_to_word[ch] = w
            used.add(w)
            print(f"  t={t} {ch}->{w!r:6} | {before} | mới: lưu {ch} -> {w!r}")
    return True

print("Kết quả:", trace_word_pattern("abba", "dog cat cat fish"))
print()
print("Kết quả:", trace_word_pattern("abba", "dog dog dog dog"))

pattern='abba', pieces=['dog', 'cat', 'cat', 'fish']
  t=0 a->'dog'  | map={}, used=[] | mới: lưu a -> 'dog'
  t=1 b->'cat'  | map={'a': 'dog'}, used=['dog'] | mới: lưu b -> 'cat'
  t=2 b->'cat'  | map={'a': 'dog', 'b': 'cat'}, used=['cat', 'dog'] | cũ: 'cat' == 'cat'? True
  t=3 a->'fish' | map={'a': 'dog', 'b': 'cat'}, used=['cat', 'dog'] | cũ: 'dog' == 'fish'? False
Kết quả: False

pattern='abba', pieces=['dog', 'dog', 'dog', 'dog']
  t=0 a->'dog'  | map={}, used=[] | mới: lưu a -> 'dog'
  t=1 b->'dog'  | map={'a': 'dog'}, used=['dog'] | mới nhưng 'dog' đã có chủ -> False
Kết quả: False


## 4. Từ #290 sang #291: mất dấu cách thì khó ở đâu?

Ở #291, `s = "redblueredblue"` **không có dấu cách**. Phần "kiểm tra song ánh" vẫn y như #290 — cái mới và khó là: **ta không biết cắt `s` ở đâu**.

Đếm thử số cách cắt. `s` dài `n` thì có `n - 1` "khe" giữa hai ký tự liên tiếp:

```text
r|e|d|b|l|u|e|r|e|d|b|l|u|e      (14 ký tự -> 13 khe)
```

Cắt `s` thành đúng `m = len(pattern)` mảnh **khác rỗng** nghĩa là chọn `m - 1` khe trong `n - 1` khe để đặt dao. Số cách là **tổ hợp chập**:

```text
C(n - 1, m - 1)        ("n-1 chọn m-1")
```

- Ví dụ 1: `n = 14`, `m = 4` → `C(13, 3) = 286` cách cắt. Cách đúng `red|blue|red|blue` là 1 trong số đó.
- Ví dụ 2 và 3: `n = 12`, `m = 4` → `C(11, 3) = 165` cách cắt.
- Trường hợp xấu nhất theo constraints (`n = 20`): lớn nhất khi `m = 10` hoặc `11` → `C(19, 9) = 92378`.

`92378` vẫn nhỏ với máy tính, nên **brute force sẽ chạy được** với `n <= 20`. Nhưng mỗi khi `n` tăng thêm một chút, con số này **tăng theo hàm mũ** — đó là lý do ta học các cách khéo hơn. `math.comb(a, b)` tính `C(a, b)` sẵn cho ta.

In [10]:
from math import comb

print("Ví dụ 1:", comb(14 - 1, 4 - 1))
print("Ví dụ 2, 3:", comb(12 - 1, 4 - 1))

n = 20
row = {m: comb(n - 1, m - 1) for m in range(1, n + 1)}
best_m = max(row, key=row.get)
print(f"n = 20: nhiều cách cắt nhất khi m = {best_m}: {row[best_m]}")
print("n = 30, m = 15:", comb(29, 14), " <- nếu đề cho dài hơn, brute force sẽ rất chậm")

Ví dụ 1: 286
Ví dụ 2, 3: 165
n = 20: nhiều cách cắt nhất khi m = 10: 92378
n = 30, m = 15: 77558760  <- nếu đề cho dài hơn, brute force sẽ rất chậm


## 5. Kiến thức nền: `itertools.combinations` — liệt kê các vị trí cắt

`combinations(iterable, k)` sinh ra **mọi cách chọn `k` phần tử** (không lặp, giữ thứ tự tăng dần) từ `iterable`.

Ví dụ `combinations([1, 2, 3, 4], 2)` sinh: `(1,2) (1,3) (1,4) (2,3) (2,4) (3,4)` — đúng `C(4, 2) = 6` bộ.

**Biểu diễn một cách cắt bằng các vị trí cắt.** Với `s = "abcde"` (`n = 5`), các khe có thể cắt là vị trí `1, 2, 3, 4` (cắt **trước** ký tự có chỉ số đó). Muốn 3 mảnh thì chọn 2 vị trí cắt:

| Vị trí cắt | Biên (thêm 0 và n) | Các mảnh |
|---|---|---|
| `(1, 2)` | `[0, 1, 2, 5]` | `["a", "b", "cde"]` |
| `(1, 3)` | `[0, 1, 3, 5]` | `["a", "bc", "de"]` |
| `(2, 4)` | `[0, 2, 4, 5]` | `["ab", "cd", "e"]` |

Từ list biên `bounds = [0, c1, c2, ..., n]`, mảnh thứ `t` là `s[bounds[t]:bounds[t + 1]]`. Vì các vị trí cắt tăng **chặt** và nằm trong `1..n-1`, mọi mảnh đều **khác rỗng** — đúng yêu cầu đề.

In [11]:
from itertools import combinations

print(list(combinations([1, 2, 3, 4], 2)))


def cut_by_positions(s: str, cuts) -> list:
    # cuts: các vị trí cắt tăng dần trong 1..len(s)-1  ->  list các mảnh khác rỗng
    bounds = [0] + list(cuts) + [len(s)]
    return [s[bounds[t]:bounds[t + 1]] for t in range(len(bounds) - 1)]


s = "abcde"
all_cuts = list(combinations(range(1, len(s)), 3 - 1))
print(f"{len(all_cuts)} cách cắt {s!r} thành 3 mảnh (C(4, 2) = {comb(4, 2)}):")
for cuts in all_cuts:
    print("  ", cuts, "->", cut_by_positions(s, cuts))

[(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
6 cách cắt 'abcde' thành 3 mảnh (C(4, 2) = 6):
   (1, 2) -> ['a', 'b', 'cde']
   (1, 3) -> ['a', 'bc', 'de']
   (1, 4) -> ['a', 'bcd', 'e']
   (2, 3) -> ['ab', 'c', 'de']
   (2, 4) -> ['ab', 'cd', 'e']
   (3, 4) -> ['abc', 'd', 'e']


## 6. Cách 1 — Brute force: thử mọi cách cắt, rồi kiểm tra song ánh

Ghép hai mảnh ghép đã có:

1. Sinh mọi bộ `m - 1` vị trí cắt bằng `combinations(range(1, n), m - 1)`.
2. Với mỗi bộ, cắt `s` thành `m` mảnh bằng `cut_by_positions`.
3. Gọi lại `matches_pieces(pattern, pieces)` của #290 để kiểm tra song ánh. Có một bộ thỏa → `True`.
4. Thử hết mà không bộ nào thỏa → `False`.

Trường hợp đặc biệt: nếu `m > n` thì không thể cắt `n` ký tự thành `m` mảnh khác rỗng → `False` ngay. (Khi đó `combinations(range(1, n), m - 1)` cũng sinh 0 bộ, nhưng viết rõ ra cho dễ đọc.)

Hàm nhận thêm tham số tùy chọn `stats` (một `dict`) để **đếm số cách cắt đã thử** — giúp ta so sánh "khối lượng công việc" với các cách sau. Khi không truyền `stats` thì hàm chạy bình thường.

Cách này chậm nhưng **gần như không thể sai** (nó làm đúng nghĩa đen của đề). Vì vậy ta dùng nó làm **trọng tài** (oracle) để kiểm tra các cách nhanh hơn ở mục 12.

In [12]:
def word_pattern_match_bruteforce(pattern: str, s: str, stats=None) -> bool:
    m, n = len(pattern), len(s)
    if m > n:
        return False
    for cuts in combinations(range(1, n), m - 1):
        if stats is not None:
            stats["cuts"] = stats.get("cuts", 0) + 1
        if matches_pieces(pattern, cut_by_positions(s, cuts)):
            return True
    return False


for pattern, s, expected in examples:
    st = {}
    got = word_pattern_match_bruteforce(pattern, s, st)
    print(f"{pattern!r:7} {s!r:17} -> {got!s:5} (expected {expected!s:5}) | đã thử {st['cuts']:3} / {comb(len(s) - 1, len(pattern) - 1)} cách cắt")
    assert got == expected

'abab'  'redblueredblue'  -> True  (expected True ) | đã thử  46 / 286 cách cắt
'aaaa'  'asdasdasdasd'    -> True  (expected True ) | đã thử  97 / 165 cách cắt
'aabb'  'xyzabcxzyabc'    -> False (expected False) | đã thử 165 / 165 cách cắt


### Cách 1 vướng ở đâu?

Độ phức tạp: có `C(n - 1, m - 1)` cách cắt, mỗi cách tốn `O(n + m)` để cắt chuỗi và kiểm tra song ánh → **Time `O(C(n - 1, m - 1) · (n + m))`**, Space `O(n + m)` cho list mảnh.

Nhưng vấn đề lớn hơn con số là **cách làm việc "ngốc"**: brute force **cắt xong cả chuỗi rồi mới hỏi**. Nhìn Ví dụ 3 `pattern = "aabb"`, `s = "xyzabcxzyabc"`:

- Hai mảnh đầu đều ứng với `a`, nên **phải bằng nhau**. Chỉ cần nhìn 2 mảnh đầu là biết cách cắt có hy vọng hay không.
- Nhưng brute force vẫn sinh đủ 4 mảnh cho **cả 165** cách cắt, rồi mới phát hiện mâu thuẫn ở mảnh thứ 2.

Cell dưới kiểm chứng: với mỗi cách cắt, ta tìm **mảnh đầu tiên gây mâu thuẫn** (chỉ số `t` nhỏ nhất làm song ánh hỏng). Nếu hầu hết đều hỏng ở `t = 1`, nghĩa là phần lớn công sức cắt các mảnh phía sau là **lãng phí**.

In [13]:
from collections import Counter


def first_conflict(pattern: str, pieces: list):
    # Trả về chỉ số mảnh đầu tiên làm song ánh hỏng (None nếu không hỏng)
    char_to_word, used = {}, set()
    for t, (ch, w) in enumerate(zip(pattern, pieces)):
        if ch in char_to_word:
            if char_to_word[ch] != w:
                return t
        else:
            if w in used:
                return t
            char_to_word[ch] = w
            used.add(w)
    return None


for pattern, s in [("aabb", "xyzabcxzyabc"), ("abab", "redblueredblue")]:
    where = Counter(first_conflict(pattern, cut_by_positions(s, cuts))
                    for cuts in combinations(range(1, len(s)), len(pattern) - 1))
    print(f"{pattern!r} / {s!r}: hỏng ở mảnh t -> số cách cắt = {dict(sorted(where.items(), key=lambda kv: (kv[0] is None, kv[0])))}")

'aabb' / 'xyzabcxzyabc': hỏng ở mảnh t -> số cách cắt = {1: 165}
'abab' / 'redblueredblue': hỏng ở mảnh t -> số cách cắt = {2: 280, None: 6}


Đọc kết quả:

- Ví dụ 3: **cả 165** cách cắt đều hỏng ngay ở mảnh `t = 1` (mảnh thứ hai, cũng là chữ `a`). Mảnh đầu chỉ có 9 độ dài khả dĩ (1..9, vì 3 mảnh sau cần ít nhất 3 ký tự), và cả 9 khả năng đều chết ngay ở mảnh thứ hai — vậy mà brute force vẫn dựng đủ 4 mảnh cho 165 bộ.
- Ví dụ 1: `None` đếm số cách cắt **hợp lệ** (không hỏng ở đâu) — có đúng **6** cách! (ta sẽ gặp lại con số 6 này ở Cách 4). **Toàn bộ** 280 cách còn lại hỏng ở `t = 2` — tức là ngay khi `a` xuất hiện lần thứ hai và không khớp với lần đầu.

**Mấu chốt cải tiến:** hãy **cắt đến đâu kiểm tra đến đó**. Ngay khi một mảnh gây mâu thuẫn, bỏ luôn mọi cách cắt có chung phần đầu đó. Hơn nữa, khi một ký tự **đã** có ánh xạ, mảnh tiếp theo của nó **bị ép** (phải đúng là chuỗi cũ) — không cần thử nhiều khả năng. Cách tổ chức "thử – kiểm tra – lùi lại" này gọi là **backtracking**. Trước hết ta học nó từ căn bản.

## 7. Kiến thức nền: đệ quy (recursion) và backtracking (quay lui)

### 7.1. Đệ quy là gì?

**Đệ quy** là khi một hàm **gọi lại chính nó** để giải một bài toán **nhỏ hơn** cùng dạng. Mỗi hàm đệ quy có 2 phần:

- **Base case (trường hợp dừng):** bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
- **Bước đệ quy:** làm một phần nhỏ công việc, rồi gọi lại hàm cho phần còn lại.

### 7.2. Backtracking là gì?

**Backtracking** là cách dùng đệ quy để **thử lần lượt các lựa chọn**. Tại mỗi bước:

1. **Chọn** một khả năng (ghi nó vào trạng thái chung, ví dụ `path.append(...)`).
2. **Đi tiếp** — gọi đệ quy để giải phần còn lại với lựa chọn đó.
3. **Gỡ** lựa chọn (xóa nó khỏi trạng thái, ví dụ `path.pop()`) để trạng thái **sạch sẽ** trước khi thử khả năng kế tiếp.

Có thể hình dung như đi trong mê cung: đến ngã rẽ thì chọn một lối, đi tới khi cụt thì **lùi lại** đúng ngã rẽ đó và thử lối khác.

### Ví dụ mini: cắt `"abcde"` thành đúng 3 mảnh bằng backtracking

Đây chính là việc `combinations` đã làm ở mục 5, nhưng giờ ta tự viết bằng đệ quy — và **chính khung này** sẽ trở thành Cách 2:

- Trạng thái: `path` = list các mảnh đã cắt; `start` = vị trí bắt đầu của mảnh tiếp theo.
- Base case: đã có đủ `k` mảnh → nếu vừa dùng hết chuỗi (`start == len(s)`) thì ghi nhận kết quả; dù sao cũng **dừng**.
- Bước đệ quy: thử mọi điểm kết thúc `end` từ `start + 1` đến `len(s)` → mảnh `s[start:end]` (luôn khác rỗng vì `end > start`).

Lưu ý `result.append(list(path))`: phải **sao chép** `path`, vì `path` sẽ còn bị `append`/`pop` tiếp; nếu lưu chính `path` thì cuối cùng mọi phần tử của `result` đều trỏ vào cùng một list (đã rỗng).

In [14]:
def split_into_k(s: str, k: int) -> list:
    result = []
    path = []

    def go(start: int) -> None:
        if len(path) == k:                      # base case: đủ k mảnh
            if start == len(s):                 # ... và dùng hết chuỗi
                result.append(list(path))       # lưu BẢN SAO của path
            return
        for end in range(start + 1, len(s) + 1):
            path.append(s[start:end])           # 1. chọn
            go(end)                             # 2. đi tiếp
            path.pop()                          # 3. gỡ

    go(0)
    return result


ways = split_into_k("abcde", 3)
print(len(ways), "cách:", ways)

# Đối chiếu với cách dùng combinations ở mục 5
by_comb = [cut_by_positions("abcde", c) for c in combinations(range(1, 5), 2)]
print("Giống combinations?", sorted(ways) == sorted(by_comb))

6 cách: [['a', 'b', 'cde'], ['a', 'bc', 'de'], ['a', 'bcd', 'e'], ['ab', 'c', 'de'], ['ab', 'cd', 'e'], ['abc', 'd', 'e']]
Giống combinations? True


### Nhìn cây gọi đệ quy: chọn → đi tiếp → gỡ

Cell dưới in cây gọi cho `"abcd"`, `k = 3` (có `C(3, 2) = 3` đáp án). Mỗi mức thụt lề là một tầng đệ quy sâu hơn.

Hai chi tiết Python cần biết:

- **Hàm lồng nhau (nested function):** `go` được định nghĩa **bên trong** `split_into_k`, nên nó **nhìn thấy** và dùng chung `path`, `result`, `s`, `k` của hàm ngoài. Vì ta chỉ **sửa nội dung** list (`append`, `pop`) chứ không gán lại tên biến, nên không cần từ khóa `nonlocal`. (Tương tự, các lời giải sau sẽ sửa `dict mapping`, `set used`, `dict stats` từ bên trong hàm lồng.)
- Ta truyền thêm `depth` (độ sâu) chỉ để in thụt lề cho dễ nhìn.

Hãy để ý các dòng `(đủ mảnh nhưng còn thừa ký tự)` và `(hết chuỗi nhưng thiếu mảnh)`: đó là những nhánh "cụt" — đã đủ 3 mảnh mà chưa dùng hết chuỗi, hoặc dùng hết chuỗi mà chưa đủ 3 mảnh. Trong 8 nhánh kết thúc (các dòng `=>`), chỉ 3 nhánh là đáp án. Backtracking đơn giản vẫn đi vào chúng; ở Cách 3 ta sẽ học cách **cắt tỉa** để khỏi đi vào.

In [15]:
def trace_split_into_k(s: str, k: int) -> None:
    path = []

    def go(start: int, depth: int) -> None:
        pad = "    " * depth
        if len(path) == k:
            status = "ĐÁP ÁN" if start == len(s) else "(đủ mảnh nhưng còn thừa ký tự)"
            print(f"{pad}=> {path} {status}")
            return
        if start == len(s):
            print(f"{pad}=> {path} (hết chuỗi nhưng thiếu mảnh)")
            return
        for end in range(start + 1, len(s) + 1):
            path.append(s[start:end])
            print(f"{pad}chọn {s[start:end]!r:6} path={path}")
            go(end, depth + 1)
            path.pop()

    go(0, 0)

trace_split_into_k("abcd", 3)

chọn 'a'    path=['a']
    chọn 'b'    path=['a', 'b']
        chọn 'c'    path=['a', 'b', 'c']
            => ['a', 'b', 'c'] (đủ mảnh nhưng còn thừa ký tự)
        chọn 'cd'   path=['a', 'b', 'cd']
            => ['a', 'b', 'cd'] ĐÁP ÁN
    chọn 'bc'   path=['a', 'bc']
        chọn 'd'    path=['a', 'bc', 'd']
            => ['a', 'bc', 'd'] ĐÁP ÁN
    chọn 'bcd'  path=['a', 'bcd']
        => ['a', 'bcd'] (hết chuỗi nhưng thiếu mảnh)
chọn 'ab'   path=['ab']
    chọn 'c'    path=['ab', 'c']
        chọn 'd'    path=['ab', 'c', 'd']
            => ['ab', 'c', 'd'] ĐÁP ÁN
    chọn 'cd'   path=['ab', 'cd']
        => ['ab', 'cd'] (hết chuỗi nhưng thiếu mảnh)
chọn 'abc'  path=['abc']
    chọn 'd'    path=['abc', 'd']
        => ['abc', 'd'] (hết chuỗi nhưng thiếu mảnh)
chọn 'abcd' path=['abcd']
    => ['abcd'] (hết chuỗi nhưng thiếu mảnh)


## 8. Cách 2 — Backtracking: gán dần từng ký tự (cách chính)

### Cách 1 vướng ở đâu → mấu chốt cải tiến

- Cách 1 **cắt xong mới kiểm tra**. Cách 2 **kiểm tra ngay khi cắt từng mảnh**: mảnh nào làm hỏng song ánh thì quay lui luôn, mọi cách cắt có chung phần đầu đó bị bỏ một lần.
- Cách 1 coi mọi mảnh là "tự do". Nhưng khi ký tự đã có ánh xạ, mảnh của nó **bị ép**: chỉ có **một** khả năng (đúng chuỗi cũ). Chỉ ký tự **mới** mới cần thử nhiều khả năng.

### Trạng thái và quyết định

Ta đi song song trên hai chuỗi:

- `i` = chỉ số ký tự đang xét trong `pattern`;
- `j` = vị trí trong `s` mà mảnh của `pattern[i]` phải **bắt đầu**;
- `mapping` (`dict`): ký tự → chuỗi đã gán; `used` (`set`): các chuỗi đã có chủ.

Hàm `backtrack(i, j)` trả lời câu hỏi: *"với `mapping`/`used` hiện tại, phần `pattern[i:]` có khớp được với phần `s[j:]` không?"*. Đáp án của bài là `backtrack(0, 0)`.

| Tình huống | Làm gì | Vì sao |
|---|---|---|
| `i == m` (hết pattern) | trả `j == n` | đã đặt xong mọi ký tự; hợp lệ chỉ khi cũng dùng **hết** `s` |
| `j == n` mà `i < m` | trả `False` | hết `s` nhưng pattern còn ký tự, mà mỗi ký tự cần mảnh khác rỗng |
| `ch = pattern[i]` **đã có** ánh xạ `word` | nếu `s.startswith(word, j)` thì gọi `backtrack(i + 1, j + len(word))`, ngược lại `False` | chiều 1 của song ánh: chỉ có **một** lựa chọn |
| `ch` **mới** | thử mọi `end` từ `j + 1` đến `n`: `cand = s[j:end]`; bỏ qua nếu `cand in used`; còn lại thì **gán** → **đi tiếp** → **gỡ** | chiều 2 của song ánh: chuỗi đã có chủ thì không được nhận |

Nếu một lời gọi con trả `True` thì ta trả `True` **ngay** (đề chỉ hỏi có tồn tại, không cần tìm thêm). Thử hết các `end` mà không được thì trả `False` — lời gọi cha sẽ gỡ lựa chọn của nó và thử tiếp.

So với ví dụ mini `split_into_k`: vòng `for end in range(j + 1, n + 1)` y hệt, chỉ thêm hai thứ: **nhánh "đã có ánh xạ"** (không thử, chỉ kiểm) và **điều kiện `cand in used`**.

In [16]:
def word_pattern_match_backtrack(pattern: str, s: str, stats=None) -> bool:
    m, n = len(pattern), len(s)
    mapping = {}      # ký tự -> chuỗi con của s
    used = set()      # các chuỗi con đã có chủ

    def backtrack(i: int, j: int) -> bool:
        if stats is not None:
            stats["calls"] = stats.get("calls", 0) + 1
        if i == m:
            return j == n
        if j == n:
            return False
        ch = pattern[i]
        if ch in mapping:                         # đã có ánh xạ: chỉ kiểm tra
            word = mapping[ch]
            if not s.startswith(word, j):
                return False
            return backtrack(i + 1, j + len(word))
        for end in range(j + 1, n + 1):           # ký tự mới: thử mọi tiền tố
            cand = s[j:end]
            if cand in used:                      # đã thuộc ký tự khác
                continue
            mapping[ch] = cand                    # 1. chọn
            used.add(cand)
            if backtrack(i + 1, end):             # 2. đi tiếp
                return True
            del mapping[ch]                       # 3. gỡ
            used.remove(cand)
        return False

    return backtrack(0, 0)


for pattern, s, expected in examples:
    got = word_pattern_match_backtrack(pattern, s)
    print(f"{pattern!r:7} {s!r:17} -> {got!s:5} (expected {expected})")
    assert got == expected

print(word_pattern_match_backtrack("ab", "xx"), "<- a='x' thì b không được nhận 'x' nữa")

'abab'  'redblueredblue'  -> True  (expected True)
'aaaa'  'asdasdasdasd'    -> True  (expected True)
'aabb'  'xyzabcxzyabc'    -> False (expected False)
False <- a='x' thì b không được nhận 'x' nữa


### Trace Cách 2: cây gọi cho `"abab"` / `"redblueredblue"` và `"ab"` / `"aa"`

Hàm `trace_backtrack` dưới đây là `word_pattern_match_backtrack` có thêm `print`. Mỗi dòng là một quyết định; thụt lề càng sâu là đệ quy càng sâu:

- `'a' thử 'r'` — ký tự **mới**, đang thử ứng viên `'r'` (kèm `mapping` **trước** khi gán);
- `'a' đã gán 'r': s[2:] bắt đầu bằng 'r'? False` — ký tự **cũ**, chỉ kiểm tra `startswith`;
- `đã bị ký tự khác dùng -> bỏ` — ứng viên bị `set used` chặn (chiều 2 của song ánh).

Ví dụ thứ hai `"ab"` / `"aa"` được chọn vì nó đi qua **cả hai** nhánh đặc biệt: bị `used` chặn, và phải **gỡ** lựa chọn của `a` để thử lại.

In [17]:
def trace_backtrack(pattern: str, s: str) -> bool:
    m, n = len(pattern), len(s)
    mapping, used = {}, set()

    def backtrack(i: int, j: int, depth: int) -> bool:
        pad = "  " * depth
        if i == m:
            print(f"{pad}i={i}, j={j}: hết pattern -> {'THÀNH CÔNG' if j == n else 'còn thừa ký tự, thất bại'}")
            return j == n
        if j == n:
            print(f"{pad}i={i}: hết s mà pattern còn ký tự -> thất bại")
            return False
        ch = pattern[i]
        if ch in mapping:
            word = mapping[ch]
            ok = s.startswith(word, j)
            print(f"{pad}i={i} '{ch}' đã gán '{word}': s[{j}:] bắt đầu bằng '{word}'? {ok}")
            return ok and backtrack(i + 1, j + len(word), depth + 1)
        for end in range(j + 1, n + 1):
            cand = s[j:end]
            if cand in used:
                print(f"{pad}i={i} '{ch}' thử '{cand}': đã bị ký tự khác dùng -> bỏ")
                continue
            print(f"{pad}i={i} '{ch}' thử '{cand}'   mapping={mapping}")
            mapping[ch] = cand
            used.add(cand)
            if backtrack(i + 1, end, depth + 1):
                return True
            del mapping[ch]
            used.remove(cand)
        return False

    ok = backtrack(0, 0, 0)
    print("Kết quả:", ok, "| mapping cuối:", mapping)
    return ok

trace_backtrack("abab", "redblueredblue")
print()
trace_backtrack("ab", "aa")

i=0 'a' thử 'r'   mapping={}
  i=1 'b' thử 'e'   mapping={'a': 'r'}
    i=2 'a' đã gán 'r': s[2:] bắt đầu bằng 'r'? False
  i=1 'b' thử 'ed'   mapping={'a': 'r'}
    i=2 'a' đã gán 'r': s[3:] bắt đầu bằng 'r'? False
  i=1 'b' thử 'edb'   mapping={'a': 'r'}
    i=2 'a' đã gán 'r': s[4:] bắt đầu bằng 'r'? False
  i=1 'b' thử 'edbl'   mapping={'a': 'r'}
    i=2 'a' đã gán 'r': s[5:] bắt đầu bằng 'r'? False
  i=1 'b' thử 'edblu'   mapping={'a': 'r'}
    i=2 'a' đã gán 'r': s[6:] bắt đầu bằng 'r'? False
  i=1 'b' thử 'edblue'   mapping={'a': 'r'}
    i=2 'a' đã gán 'r': s[7:] bắt đầu bằng 'r'? True
      i=3 'b' đã gán 'edblue': s[8:] bắt đầu bằng 'edblue'? True
        i=4, j=14: hết pattern -> THÀNH CÔNG
Kết quả: True | mapping cuối: {'a': 'r', 'b': 'edblue'}

i=0 'a' thử 'a'   mapping={}
  i=1 'b' thử 'a': đã bị ký tự khác dùng -> bỏ
i=0 'a' thử 'aa'   mapping={}
  i=1: hết s mà pattern còn ký tự -> thất bại
Kết quả: False | mapping cuối: {}


False

### Đọc trace `"abab"` / `"redblueredblue"` bằng bảng

Chỉ số: `r0 e1 d2 b3 l4 u5 e6 r7 e8 d9 b10 l11 u12 e13` (`n = 14`).

| Bước | `i` | `j` | Ký tự | Trạng thái | Hành động | Kết quả |
|---:|---:|---:|---|---|---|---|
| 1 | 0 | 0 | `a` | mới | thử `a = 'r'` (ứng viên ngắn nhất) | đi tiếp |
| 2 | 1 | 1 | `b` | mới | thử `b = 'e'` → con: `a` phải bắt đầu ở `j = 2`, nhưng `s[2] = 'd'` | ✗, gỡ `b` |
| 3 | 1 | 1 | `b` | mới | thử `'ed'`, `'edb'`, `'edbl'`, `'edblu'` → `a` phải ở `j = 3, 4, 5, 6`: gặp `'b'`, `'l'`, `'u'`, `'e'` | đều ✗ |
| 4 | 1 | 1 | `b` | mới | thử `b = 'edblue'` → `a` ở `j = 7`: `s[7] = 'r'` | ✓ |
| 5 | 3 | 8 | `b` | đã gán `'edblue'` | `s[8:14] = 'edblue'` | ✓ |
| 6 | 4 | 14 | — | hết pattern, `j == n` | | **True** |

**Bất ngờ:** thuật toán tìm ra `a -> "r"`, `b -> "edblue"` chứ **không** phải `red` / `blue` như đề minh họa! Kiểm tra lại: `"r" + "edblue" + "r" + "edblue" = "redblueredblue"` ✓, và `"r" != "edblue"` ✓. Đây là một ánh xạ **hoàn toàn hợp lệ**. Đề chỉ hỏi **có tồn tại** hay không, nên tìm được **một** ánh xạ bất kỳ là đủ — đó là lý do ta được phép `return True` ngay khi lời gọi con thành công. Vì `a` thử ứng viên **ngắn nhất trước**, nó tìm ra ánh xạ có `a` ngắn nhất.

**Trace `"ab"` / `"aa"`:** `a = 'a'` → đến `b` ở `j = 1`, ứng viên duy nhất là `'a'` nhưng `'a'` đã thuộc `a` → bị `used` chặn. Quay lui, **gỡ** `a = 'a'`, thử `a = 'aa'` → hết chuỗi mà `b` chưa có mảnh → thất bại. Hết lựa chọn → `False`. Nếu không có `set used`, thuật toán sẽ chấp nhận `a = b = 'a'` và trả `True` sai (xem Lỗi 1 ở mục 15).

In [18]:
found = {"a": "r", "b": "edblue"}
print(apply_mapping("abab", found))
print("Ánh xạ tìm được hợp lệ?", is_valid_mapping("abab", "redblueredblue", found))
print("Ánh xạ trong đề hợp lệ? ", is_valid_mapping("abab", "redblueredblue", {"a": "red", "b": "blue"}))

redblueredblue
Ánh xạ tìm được hợp lệ? True
Ánh xạ trong đề hợp lệ?  True


### Độ phức tạp Cách 2 — và nó bớt việc bao nhiêu so với Cách 1?

Ký hiệu: `m = len(pattern)`, `n = len(s)`, `k` = số ký tự **khác nhau** trong `pattern` (ví dụ `"abab"` có `k = 2`).

- Chỉ ký tự **mới** mới tạo nhánh, và mỗi ký tự mới có **tối đa `n`** ứng viên (độ dài 1..n). Ký tự cũ chỉ có **một** đường đi. Vì vậy cây tìm kiếm có tối đa khoảng **`n^k` lá**; mỗi đường từ gốc tới lá dài tối đa `m` lời gọi, mỗi lời gọi tốn `O(n)` để cắt/so chuỗi. Cận trên thô: **Time `O(n^k · m · n)`** — vẫn **hàm mũ** trong trường hợp xấu nhất, nhưng mũ theo `k` (số ký tự khác nhau) chứ không phải theo `m`, và trên thực tế nhỏ hơn rất nhiều vì các nhánh hỏng bị bỏ rất sớm.
- **Space `O(m + n)`:** độ sâu đệ quy tối đa `m + 1`; `mapping`/`used` chứa các mảnh **rời nhau** của `s[:j]` nên tổng độ dài các chuỗi trong đó `<= n`.

Đếm thực tế: cell dưới so **số cách cắt** Cách 1 phải xét với **số lời gọi** `backtrack` của Cách 2 (đơn vị khác nhau, nhưng mỗi đơn vị đều tốn cỡ `O(n)` việc, nên so sánh được về độ lớn).

In [19]:
cases = [
    ("abab", "redblueredblue"),
    ("aaaa", "asdasdasdasd"),
    ("aabb", "xyzabcxzyabc"),
    ("abba", "dogcatcatdog"),
    ("abba", "dogdogdogdog"),
    ("abababab", "xy" * 9 + "yx"),
    ("abcdabcdab", "xyzw" * 5),
]

print(f"{'pattern':12} {'s':22} {'kết quả':8} {'Cách 1: cách cắt':>17} {'Cách 2: lời gọi':>16}")
for pattern, s in cases:
    st1, st2 = {}, {}
    r1 = word_pattern_match_bruteforce(pattern, s, st1)
    r2 = word_pattern_match_backtrack(pattern, s, st2)
    assert r1 == r2
    print(f"{pattern:12} {s:22} {r1!s:8} {st1['cuts']:>17} {st2['calls']:>16}")

pattern      s                      kết quả   Cách 1: cách cắt  Cách 2: lời gọi
abab         redblueredblue         True                    46               10
aaaa         asdasdasdasd           True                    97                7
aabb         xyzabcxzyabc           False                  165               13
abba         dogcatcatdog           True                    97               30
abba         dogdogdogdog           False                  165               82


abababab     xyxyxyxyxyxyxyxyxyyx   False                50388              279


abcdabcdab   xyzwxyzwxyzwxyzwxyzw   True                 34843              476


## 9. Cách 3 — Backtracking + cắt tỉa (pruning) theo độ dài

### Cách 2 còn vướng ở đâu?

Xem trace Cách 2 trên Ví dụ 3 (`"aabb"` / `"xyzabcxzyabc"`, `n = 12`). Ký tự `a` thử **mọi** độ dài từ 1 đến 12 — kể cả `a = "xyzabcxzyabc"` (nuốt cả chuỗi!). Nhưng suy nghĩ một chút là thấy vô lý: `a` xuất hiện **2 lần**, `b` cũng cần ít nhất 2 ký tự (2 lần, mỗi lần ≥ 1), nên `2 · len(a) + 2 <= 12`, tức `len(a) <= 5`. Mọi ứng viên dài 6..12 đều **chắc chắn hỏng** mà vẫn bị thử.

In [20]:
trace_backtrack("aabb", "xyzabcxzyabc")

i=0 'a' thử 'x'   mapping={}
  i=1 'a' đã gán 'x': s[1:] bắt đầu bằng 'x'? False
i=0 'a' thử 'xy'   mapping={}
  i=1 'a' đã gán 'xy': s[2:] bắt đầu bằng 'xy'? False
i=0 'a' thử 'xyz'   mapping={}
  i=1 'a' đã gán 'xyz': s[3:] bắt đầu bằng 'xyz'? False
i=0 'a' thử 'xyza'   mapping={}
  i=1 'a' đã gán 'xyza': s[4:] bắt đầu bằng 'xyza'? False
i=0 'a' thử 'xyzab'   mapping={}
  i=1 'a' đã gán 'xyzab': s[5:] bắt đầu bằng 'xyzab'? False
i=0 'a' thử 'xyzabc'   mapping={}
  i=1 'a' đã gán 'xyzabc': s[6:] bắt đầu bằng 'xyzabc'? False
i=0 'a' thử 'xyzabcx'   mapping={}
  i=1 'a' đã gán 'xyzabcx': s[7:] bắt đầu bằng 'xyzabcx'? False
i=0 'a' thử 'xyzabcxz'   mapping={}
  i=1 'a' đã gán 'xyzabcxz': s[8:] bắt đầu bằng 'xyzabcxz'? False
i=0 'a' thử 'xyzabcxzy'   mapping={}
  i=1 'a' đã gán 'xyzabcxzy': s[9:] bắt đầu bằng 'xyzabcxzy'? False
i=0 'a' thử 'xyzabcxzya'   mapping={}
  i=1 'a' đã gán 'xyzabcxzya': s[10:] bắt đầu bằng 'xyzabcxzya'? False
i=0 'a' thử 'xyzabcxzyab'   mapping={}
  i=1 'a' đã gá

False

### Ba mức cắt tỉa, từ thô đến tinh

**Mức 1 — đếm thô:** mỗi ký tự còn lại của `pattern` cần **ít nhất 1** ký tự của `s`. Vậy phải có

```text
n - j  >=  m - i        (số ký tự s còn lại  >=  số ký tự pattern còn lại)
```

Ví dụ: `pattern = "abcd"`, đang ở `i = 1` (còn 3 ký tự `bcd`), `s` còn lại 2 ký tự → không thể, quay lui ngay.

**Mức 2 — dùng độ dài đã biết:** nếu một ký tự trong phần còn lại **đã được gán**, nó cần **đúng** `len(mapping[c])` ký tự chứ không chỉ 1. Định nghĩa

```text
min_need(i) = tổng, với mỗi c trong pattern[i:], của:  len(mapping[c]) nếu c đã gán, ngược lại 1
```

và yêu cầu `min_need(i) <= n - j`. Mức 1 là trường hợp riêng của mức 2 khi chưa ký tự nào trong phần còn lại được gán. Ví dụ: `pattern = "abba"`, `a = "dog"` đã gán, đang ở `i = 1`: phần còn lại `"bba"` cần ít nhất `1 + 1 + 3 = 5` ký tự.

**Mức 3 — chặn trên cho độ dài ứng viên của ký tự mới:** giả sử `ch = pattern[i]` là ký tự mới, còn xuất hiện `cnt = pattern.count(ch, i)` lần, và ta định cho nó độ dài `L`. Các vị trí **khác** `ch` trong phần còn lại cần ít nhất `others = min_need(i) - cnt` ký tự (vì `min_need` đã tính mỗi lần xuất hiện của `ch` là 1). Vậy phải có

```text
cnt · L + others  <=  n - j        =>        L  <=  (n - j - others) // cnt
```

Ví dụ 3 ở gốc (`i = 0`, `j = 0`): `cnt = 2` (hai chữ `a`), `others = 2` (hai chữ `b`), nên `L <= (12 - 0 - 2) // 2 = 5`. Thay vì 12 ứng viên, `a` chỉ còn **5**.

Cell dưới viết `min_need` và `max_len_for_new` thành hàm riêng (nhận `mapping` làm tham số) để ta tính thử vài con số trước khi ghép vào thuật toán.

In [21]:
def min_need(pattern: str, i: int, mapping: dict) -> int:
    # Số ký tự tối thiểu mà pattern[i:] cần, với các ánh xạ hiện có
    return sum(len(mapping[c]) if c in mapping else 1 for c in pattern[i:])


def max_len_for_new(pattern: str, i: int, j: int, n: int, mapping: dict) -> int:
    # Độ dài tối đa có thể cho ký tự MỚI pattern[i] khi nó bắt đầu ở vị trí j
    ch = pattern[i]
    cnt = pattern.count(ch, i)
    others = min_need(pattern, i, mapping) - cnt
    return (n - j - others) // cnt


print("Mức 1: 'abcd', i=1, còn 2 ký tự ->", "cắt tỉa" if min_need("abcd", 1, {}) > 2 else "đi tiếp")
print("Mức 2: 'abba', a='dog', i=1 cần ít nhất", min_need("abba", 1, {"a": "dog"}), "ký tự")
print("Mức 3: Ví dụ 3 ở gốc, len(a) <=", max_len_for_new("aabb", 0, 0, 12, {}))
print("Mức 3: 'abba'/'dogcatcatdog', a='dog', b bắt đầu ở j=3: len(b) <=",
      max_len_for_new("abba", 1, 3, 12, {"a": "dog"}))

Mức 1: 'abcd', i=1, còn 2 ký tự -> cắt tỉa
Mức 2: 'abba', a='dog', i=1 cần ít nhất 5 ký tự
Mức 3: Ví dụ 3 ở gốc, len(a) <= 5
Mức 3: 'abba'/'dogcatcatdog', a='dog', b bắt đầu ở j=3: len(b) <= 3


Dòng cuối: với `a = "dog"`, phần còn lại `"bba"` phải lấp đúng 9 ký tự `s[3:]`; `a` cuối chiếm 3, nên hai chữ `b` có chung 6 ký tự → `len(b) <= 3`. Đúng: `b = "cat"` có độ dài 3.

### Ghép lại: code Cách 3

So với Cách 2, chỉ thêm hai chỗ:

1. Đầu mỗi lời gọi (sau base case `i == m`): tính `need = min_need(i)`; nếu `need > n - j` thì trả `False` (mức 1 + 2). Kiểm tra này cũng thay luôn base case `j == n` của Cách 2 (vì khi `i < m`, `need >= 1 > 0 = n - j`) và tự xử lý trường hợp `m > n` ngay ở gốc.
2. Với ký tự mới: thay vì `for end in range(j + 1, n + 1)`, chỉ thử `L` từ 1 đến `max_len` (mức 3), ứng viên là `s[j:j + L]`.

Tính `min_need` mất `O(m)` mỗi lời gọi; với `m <= 20` thì rất rẻ so với số nhánh tiết kiệm được.

In [22]:
def word_pattern_match_pruned(pattern: str, s: str, stats=None) -> bool:
    m, n = len(pattern), len(s)
    mapping, used = {}, set()

    def backtrack(i: int, j: int) -> bool:
        if stats is not None:
            stats["calls"] = stats.get("calls", 0) + 1
        if i == m:
            return j == n
        need = sum(len(mapping[c]) if c in mapping else 1 for c in pattern[i:])
        if need > n - j:                          # mức 1 + 2
            return False
        ch = pattern[i]
        if ch in mapping:
            word = mapping[ch]
            if not s.startswith(word, j):
                return False
            return backtrack(i + 1, j + len(word))
        cnt = pattern.count(ch, i)
        max_len = (n - j - (need - cnt)) // cnt   # mức 3
        for L in range(1, max_len + 1):
            cand = s[j:j + L]
            if cand in used:
                continue
            mapping[ch] = cand
            used.add(cand)
            if backtrack(i + 1, j + L):
                return True
            del mapping[ch]
            used.remove(cand)
        return False

    return backtrack(0, 0)


for pattern, s, expected in examples:
    got = word_pattern_match_pruned(pattern, s)
    print(f"{pattern!r:7} {s!r:17} -> {got!s:5} (expected {expected})")
    assert got == expected
print(word_pattern_match_pruned("abcde", "xyz"), "<- m > n: bị cắt tỉa ngay ở gốc")

'abab'  'redblueredblue'  -> True  (expected True)
'aaaa'  'asdasdasdasd'    -> True  (expected True)
'aabb'  'xyzabcxzyabc'    -> False (expected False)
False <- m > n: bị cắt tỉa ngay ở gốc


### Trace Cách 3 trên Ví dụ 3: từ 25 dòng xuống 12 dòng

So với trace Cách 2 ở đầu mục này, `a` giờ chỉ thử 5 ứng viên. Mỗi ứng viên hỏng ngay ở lần xuất hiện thứ hai của `a` (vì `s[0:L] != s[L:2L]` với mọi `L = 1..5`), nên thuật toán kết luận `False` sau **6** lời gọi thay vì 13.

Dòng đầu của trace in luôn phép tính mức 3: `xuất hiện 2 lần, phần khác cần >= 2 -> độ dài <= 5`.

In [23]:
def trace_pruned(pattern: str, s: str) -> bool:
    m, n = len(pattern), len(s)
    mapping, used = {}, set()

    def backtrack(i: int, j: int, depth: int) -> bool:
        pad = "  " * depth
        if i == m:
            print(f"{pad}i={i}, j={j}: hết pattern -> {'THÀNH CÔNG' if j == n else 'thất bại'}")
            return j == n
        need = sum(len(mapping[c]) if c in mapping else 1 for c in pattern[i:])
        if need > n - j:
            print(f"{pad}i={i}, j={j}: cần >= {need} ký tự nhưng chỉ còn {n - j} -> cắt tỉa")
            return False
        ch = pattern[i]
        if ch in mapping:
            word = mapping[ch]
            ok = s.startswith(word, j)
            print(f"{pad}i={i} '{ch}' đã gán '{word}': s[{j}:] bắt đầu bằng '{word}'? {ok}")
            return ok and backtrack(i + 1, j + len(word), depth + 1)
        cnt = pattern.count(ch, i)
        max_len = (n - j - (need - cnt)) // cnt
        print(f"{pad}i={i} '{ch}' mới: xuất hiện {cnt} lần, phần khác cần >= {need - cnt} -> độ dài <= {max_len}")
        for L in range(1, max_len + 1):
            cand = s[j:j + L]
            if cand in used:
                print(f"{pad}  thử '{cand}': đã bị ký tự khác dùng -> bỏ")
                continue
            print(f"{pad}  thử '{ch}' = '{cand}'")
            mapping[ch] = cand
            used.add(cand)
            if backtrack(i + 1, j + L, depth + 2):
                return True
            del mapping[ch]
            used.remove(cand)
        return False

    ok = backtrack(0, 0, 0)
    print("Kết quả:", ok)
    return ok

trace_pruned("aabb", "xyzabcxzyabc")

i=0 'a' mới: xuất hiện 2 lần, phần khác cần >= 2 -> độ dài <= 5
  thử 'a' = 'x'
    i=1 'a' đã gán 'x': s[1:] bắt đầu bằng 'x'? False
  thử 'a' = 'xy'
    i=1 'a' đã gán 'xy': s[2:] bắt đầu bằng 'xy'? False
  thử 'a' = 'xyz'
    i=1 'a' đã gán 'xyz': s[3:] bắt đầu bằng 'xyz'? False
  thử 'a' = 'xyza'
    i=1 'a' đã gán 'xyza': s[4:] bắt đầu bằng 'xyza'? False
  thử 'a' = 'xyzab'
    i=1 'a' đã gán 'xyzab': s[5:] bắt đầu bằng 'xyzab'? False
Kết quả: False


False

### Cắt tỉa giúp bao nhiêu? — và vì sao nó **không bao giờ** bỏ sót đáp án

**Không bỏ sót:** giả sử có một ánh xạ hợp lệ kéo dài trạng thái hiện tại. Khi đó `s[j:]` đúng bằng phần nối các chuỗi của `pattern[i:]`; ký tự đã gán chiếm đúng độ dài của nó, ký tự chưa gán chiếm **ít nhất 1** mỗi lần → `n - j >= min_need(i)`. Tương tự, nếu `ch` mới có độ dài thật là `L` thì `n - j = cnt · L + (độ dài thật của phần khác) >= cnt · L + others` → `L <= max_len`. Vậy mọi nhánh bị cắt đều là nhánh **không chứa** ánh xạ hợp lệ nào. Cắt tỉa chỉ bỏ việc thừa, không đổi kết quả.

**Tiết kiệm bao nhiêu:** cell dưới đếm số lời gọi của Cách 2 và Cách 3. Ba dòng cuối là các input "khó" với `n = 20` mà kết quả là `False` (phải duyệt hết cây mới dám kết luận).

Để ý dòng `"abcdefg"` / `"x" * 20`: cắt tỉa giúp ít hơn hẳn. Lý do: 7 ký tự khác nhau phải nhận 7 chuỗi `x...x` **khác nhau**, tức độ dài khác nhau, nên tổng ít nhất `1 + 2 + ... + 7 = 28 > 20` → `False`. Lý do thất bại là **song ánh**, mà cắt tỉa của ta chỉ tính **độ dài tối thiểu 1** cho mỗi ký tự chưa gán, nên nó không "nhìn thấy" lý do này sớm. Cắt tỉa mạnh nhất khi `pattern` có ký tự **lặp lại nhiều lần**.

In [24]:
hard_cases = cases + [
    ("abcabcabcd", "x" * 20),
    ("ababababab", "x" * 19 + "y"),
    ("abcdefg", "x" * 20),
]

print(f"{'pattern':12} {'s':22} {'kết quả':8} {'Cách 2':>7} {'Cách 3':>7}")
for pattern, s in hard_cases:
    st2, st3 = {}, {}
    r2 = word_pattern_match_backtrack(pattern, s, st2)
    r3 = word_pattern_match_pruned(pattern, s, st3)
    assert r2 == r3
    print(f"{pattern:12} {s:22} {r2!s:8} {st2['calls']:>7} {st3['calls']:>7}")

pattern      s                      kết quả   Cách 2  Cách 3
abab         redblueredblue         True          10      10
aaaa         asdasdasdasd           True           7       7
aabb         xyzabcxzyabc           False         13       6
abba         dogcatcatdog           True          30      18
abba         dogdogdogdog           False         82      22
abababab     xyxyxyxyxyxyxyxyxyyx   False        279      32
abcdabcdab   xyzwxyzwxyzwxyzwxyzw   True         476     110
abcabcabcd   xxxxxxxxxxxxxxxxxxxx   False       1731      55
ababababab   xxxxxxxxxxxxxxxxxxxy   False        361      38
abcdefg      xxxxxxxxxxxxxxxxxxxx   False       5619    2267


## 10. Cách 4 (tùy chọn) — Đoán độ dài trước, kiểm tra sau

### Ý tưởng: một khi biết độ dài, mọi thứ bị ép

Quan sát: nếu ta biết **độ dài** chuỗi của mỗi ký tự khác nhau, thì cách cắt `s` hoàn toàn xác định — chỉ việc đi dọc `pattern` và cắt đúng độ dài. Nên thay vì thử chuỗi, ta thử **bộ độ dài**.

Gọi `count[c]` là số lần ký tự `c` xuất hiện trong `pattern`, `len_c` là độ dài chuỗi của `c`. Vì nối lại phải đúng bằng `s`:

```text
Σ  count[c] · len_c  =  n          (với mọi ký tự khác nhau c, và len_c >= 1)
```

Ví dụ 1: `pattern = "abab"` → `count = {a: 2, b: 2}`, `n = 14` → `2·len_a + 2·len_b = 14` → `len_a + len_b = 7` → có đúng **6** nghiệm `(1,6), (2,5), (3,4), (4,3), (5,2), (6,1)`. Mỗi nghiệm cho một cách cắt; ta kiểm tra cách cắt đó có nhất quán và song ánh không.

### Công cụ mới: `Counter` và `itertools.product`

- `Counter("abab")` đếm tần suất: `Counter({'a': 2, 'b': 2})`. Từ Python 3.7, thứ tự key là **thứ tự xuất hiện đầu tiên**, nên `list(Counter("abab")) == ['a', 'b']`.
- `product(range(1, 15), repeat=2)` sinh **mọi cặp** `(x, y)` với `x, y` trong `1..14` — tức `14 · 14 = 196` cặp (giống hai vòng `for` lồng nhau).

Cell dưới sinh mọi bộ độ dài bằng `product`, lọc theo phương trình, rồi dựng ánh xạ cho mỗi bộ bằng hàm `build_mapping` (trả `None` nếu cùng một ký tự gặp hai mảnh khác nhau).

In [25]:
from itertools import product


def build_mapping(pattern: str, s: str, lengths: dict):
    # Cắt s theo độ dài đã định cho từng ký tự; trả về mapping, hoặc None nếu mâu thuẫn
    mapping, j = {}, 0
    for ch in pattern:
        piece = s[j:j + lengths[ch]]
        if mapping.setdefault(ch, piece) != piece:   # ký tự cũ gặp mảnh khác
            return None
        j += lengths[ch]
    return mapping


pattern, s = "abab", "redblueredblue"
cnt = Counter(pattern)
chars = list(cnt)
print(cnt, chars)

total = 0
for lens in product(range(1, len(s) + 1), repeat=len(chars)):
    total += 1
    if sum(cnt[c] * L for c, L in zip(chars, lens)) == len(s):
        mp = build_mapping(pattern, s, dict(zip(chars, lens)))
        print(f"  độ dài {lens} -> {mp}  hợp lệ? {mp is not None and is_valid_mapping(pattern, s, mp)}")
print(f"product sinh {total} bộ, chỉ những bộ thỏa phương trình được in ra")

Counter({'a': 2, 'b': 2}) ['a', 'b']
  độ dài (1, 6) -> {'a': 'r', 'b': 'edblue'}  hợp lệ? True
  độ dài (2, 5) -> {'a': 're', 'b': 'dblue'}  hợp lệ? True
  độ dài (3, 4) -> {'a': 'red', 'b': 'blue'}  hợp lệ? True
  độ dài (4, 3) -> {'a': 'redb', 'b': 'lue'}  hợp lệ? True
  độ dài (5, 2) -> {'a': 'redbl', 'b': 'ue'}  hợp lệ? True
  độ dài (6, 1) -> {'a': 'redblu', 'b': 'e'}  hợp lệ? True
product sinh 196 bộ, chỉ những bộ thỏa phương trình được in ra


Ví dụ 1 có **6 ánh xạ hợp lệ** — khớp với con số `None: 6` ở mục 6. Chú ý `mapping.setdefault(ch, piece)`: nếu `ch` chưa có thì gán `piece` và trả về `piece`; nếu đã có thì trả về giá trị cũ — gọn để kiểm tra "ký tự cũ phải gặp lại đúng mảnh cũ".

### `product` nổ khi có nhiều ký tự khác nhau → đệ quy có "ngân sách"

`product` sinh `n^k` bộ (với `k` ký tự khác nhau): `k = 7`, `n = 20` → `20^7 = 1 280 000 000` bộ — quá nhiều, dù chỉ rất ít bộ thỏa phương trình. Cách sửa: chọn độ dài **từng ký tự một** bằng đệ quy, mang theo **ngân sách** `budget` = số ký tự của `s` chưa được phân:

- Chọn `len` cho ký tự thứ `t` là `c`: mỗi ký tự **sau** nó cần ít nhất `count` ký tự (độ dài ≥ 1), gọi tổng đó là `rest_min`. Nên `count[c] · len <= budget - rest_min` → `len` chạy từ 1 đến `(budget - rest_min) // count[c]`. (Cùng tinh thần cắt tỉa mức 3!)
- Khi đã chọn hết `k` độ dài: chỉ chấp nhận nếu `budget == 0` (dùng vừa khít `s`), rồi `build_mapping` và kiểm tra các chuỗi **khác nhau** (`len(set(values)) == k`).

**Độ phức tạp Cách 4:** mỗi bộ độ dài đầy đủ được quyết định bởi `k - 1` độ dài đầu (độ dài cuối bị ép bởi phương trình), và tổng của chúng `<= n - 1`, nên số bộ **`<= C(n - 1, k - 1)`**; mỗi bộ kiểm tra mất `O(m + n)`. Time cỡ **`O(C(n - 1, k - 1) · (m + n))`**, Space `O(k + n)`. Cách này đặc biệt mạnh khi `k` nhỏ (ví dụ `k = 1`: chỉ có tối đa 1 bộ!), nhưng nó **không** tận dụng việc "kiểm tra sớm" như backtracking — mọi bộ độ dài đều được dựng đủ rồi mới kiểm tra.

**Đoán trước khi chạy:** với `"abcdefg"` / `"x" * 20` (`k = 7`, mỗi ký tự xuất hiện 1 lần), Cách 4 phải kiểm tra bao nhiêu bộ? Mọi bộ 7 số nguyên dương có tổng 20 đều thỏa phương trình, và có đúng `C(19, 6) = 27132` bộ như vậy — **chạm đúng cận trên**. Ngược lại, với `pattern` có ký tự lặp nhiều (như `"ababababab"`), chỉ còn vài bộ.

In [26]:
def word_pattern_match_lengths(pattern: str, s: str, stats=None) -> bool:
    n = len(s)
    count = Counter(pattern)
    chars = list(count)               # các ký tự khác nhau, theo thứ tự xuất hiện
    k = len(chars)
    lengths = {}

    def assign(t: int, budget: int) -> bool:
        if t == k:
            if budget != 0:
                return False
            if stats is not None:
                stats["tuples"] = stats.get("tuples", 0) + 1
            mp = build_mapping(pattern, s, lengths)
            return mp is not None and len(set(mp.values())) == k
        c = chars[t]
        rest_min = sum(count[d] for d in chars[t + 1:])
        for L in range(1, (budget - rest_min) // count[c] + 1):
            lengths[c] = L
            if assign(t + 1, budget - count[c] * L):
                return True
        return False

    return assign(0, n)


for pattern, s in hard_cases:
    st = {}
    got = word_pattern_match_lengths(pattern, s, st)
    assert got == word_pattern_match_bruteforce(pattern, s)
    print(f"{pattern:12} {s:22} -> {got!s:5} | số bộ độ dài đầy đủ đã kiểm tra: {st.get('tuples', 0)}")

abab         redblueredblue         -> True  | số bộ độ dài đầy đủ đã kiểm tra: 1
aaaa         asdasdasdasd           -> True  | số bộ độ dài đầy đủ đã kiểm tra: 1
aabb         xyzabcxzyabc           -> False | số bộ độ dài đầy đủ đã kiểm tra: 5
abba         dogcatcatdog           -> True  | số bộ độ dài đầy đủ đã kiểm tra: 3
abba         dogdogdogdog           -> False | số bộ độ dài đầy đủ đã kiểm tra: 5


abababab     xyxyxyxyxyxyxyxyxyyx   -> False | số bộ độ dài đầy đủ đã kiểm tra: 4


abcdabcdab   xyzwxyzwxyzwxyzwxyzw   -> True  | số bộ độ dài đầy đủ đã kiểm tra: 8


abcabcabcd   xxxxxxxxxxxxxxxxxxxx   -> False | số bộ độ dài đầy đủ đã kiểm tra: 20


ababababab   xxxxxxxxxxxxxxxxxxxy   -> False | số bộ độ dài đầy đủ đã kiểm tra: 3


abcdefg      xxxxxxxxxxxxxxxxxxxx   -> False | số bộ độ dài đầy đủ đã kiểm tra: 27132


## 11. Code nộp LeetCode — Cách 2 + cắt tỉa của Cách 3

Ta nộp **backtracking** (Cách 2) có thêm **cắt tỉa theo độ dài** (Cách 3):

- Giữ đúng signature `def wordPatternMatch(self, pattern: str, s: str) -> bool`.
- `mapping` và `used` được tạo **mới** trong mỗi lần gọi `wordPatternMatch` → các test không ảnh hưởng nhau.
- Hàm con `backtrack(i, j)` lồng bên trong, dùng chung `mapping`, `used`, `pattern`, `s`.
- Không còn `print`, không còn `stats` — code nộp nên gọn.

Với `n <= 20`, bản Cách 2 thuần (không cắt tỉa) thường cũng đủ nhanh; phần cắt tỉa là "bảo hiểm" cho các input khó và là điểm cộng khi phỏng vấn.

In [27]:
class Solution:
    def wordPatternMatch(self, pattern: str, s: str) -> bool:
        m, n = len(pattern), len(s)
        mapping = {}      # ký tự pattern -> chuỗi con khác rỗng của s
        used = set()      # các chuỗi con đã có chủ (chiều ngược của song ánh)

        def backtrack(i: int, j: int) -> bool:
            if i == m:                                   # đặt xong mọi ký tự
                return j == n                            # ... và phải dùng hết s
            need = 0                                     # số ký tự tối thiểu pattern[i:] cần
            for c in pattern[i:]:
                need += len(mapping[c]) if c in mapping else 1
            if need > n - j:                             # cắt tỉa theo độ dài
                return False

            ch = pattern[i]
            if ch in mapping:                            # đã có ánh xạ: chỉ kiểm tra
                word = mapping[ch]
                if not s.startswith(word, j):
                    return False
                return backtrack(i + 1, j + len(word))

            cnt = pattern.count(ch, i)                   # ch còn xuất hiện cnt lần
            max_len = (n - j - (need - cnt)) // cnt
            for length in range(1, max_len + 1):
                cand = s[j:j + length]
                if cand in used:                         # chuỗi đã thuộc ký tự khác
                    continue
                mapping[ch] = cand                       # chọn
                used.add(cand)
                if backtrack(i + 1, j + length):         # đi tiếp
                    return True
                del mapping[ch]                          # gỡ
                used.remove(cand)
            return False

        return backtrack(0, 0)

### Bộ test bằng `assert`

Ngoài 3 ví dụ của đề, ta kiểm tra các nhóm dễ sai:

1. **Nhỏ nhất:** `pattern` và `s` đều 1 ký tự (`"a"` / `"x"` → `True`: `a -> "x"`).
2. **Song ánh bị vi phạm:** `"ab"` / `"aa"`, `"ab"` / `"xx"` → `False` (hai ký tự không được chung chuỗi).
3. **`pattern` dài hơn `s`:** `"abc"` / `"ab"` → `False`.
4. **Một ký tự lặp:** `"aa"` / `"xyx"` → `False` (độ dài lẻ không chia đôi được); `"aaaa"` / `"aaaa"` → `True`.
5. **Ánh xạ "lạ" vẫn hợp lệ:** `"ab"` / `"aaa"` → `True` (`a -> "a"`, `b -> "aa"`); `"abab"` / `"asdasdasdasd"` → `True` (`a -> "a"`, `b -> "sdasd"`).
6. **Không dùng hết `s`:** `"aba"` / `"xyxz"` → `False` (với `a -> "x"`, `b -> "y"` thì khớp được `"xyx"` nhưng còn thừa `"z"`).
7. **Gần biên lớn:** `"a"` / `"x" * 20`, `"abcdefghij"` / 20 ký tự, các input khó ở mục 9.

In [28]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}")

sol = Solution()
for idx, (pattern, s, expected) in enumerate(examples, 1):
    check(f"ví dụ {idx}", sol.wordPatternMatch(pattern, s), expected)

check("1 ký tự", sol.wordPatternMatch("a", "x"), True)
check("song ánh: 'ab'/'aa'", sol.wordPatternMatch("ab", "aa"), False)
check("song ánh: 'ab'/'xx'", sol.wordPatternMatch("ab", "xx"), False)
check("pattern dài hơn s", sol.wordPatternMatch("abc", "ab"), False)
check("độ dài lẻ không chia đôi", sol.wordPatternMatch("aa", "xyx"), False)
check("'aaaa'/'aaaa'", sol.wordPatternMatch("aaaa", "aaaa"), True)
check("'ab'/'aaa'", sol.wordPatternMatch("ab", "aaa"), True)
check("'abab'/'asdasdasdasd'", sol.wordPatternMatch("abab", "asdasdasdasd"), True)
check("thừa ký tự cuối", sol.wordPatternMatch("aba", "xyxz"), False)
check("'abba'/'dogcatcatdog'", sol.wordPatternMatch("abba", "dogcatcatdog"), True)
check("'abba'/'dogdogdogdog'", sol.wordPatternMatch("abba", "dogdogdogdog"), False)
check("1 ký tự nuốt cả s dài 20", sol.wordPatternMatch("a", "x" * 20), True)
check("10 ký tự khác nhau, s dài 20", sol.wordPatternMatch("abcdefghij", "abcdefghijklmnopqrst"), True)
for pattern, s in hard_cases:
    check(f"khó: {pattern}/{s}", sol.wordPatternMatch(pattern, s), word_pattern_match_bruteforce(pattern, s))

PASS  ví dụ 1
PASS  ví dụ 2
PASS  ví dụ 3
PASS  1 ký tự
PASS  song ánh: 'ab'/'aa'
PASS  song ánh: 'ab'/'xx'
PASS  pattern dài hơn s
PASS  độ dài lẻ không chia đôi
PASS  'aaaa'/'aaaa'
PASS  'ab'/'aaa'
PASS  'abab'/'asdasdasdasd'
PASS  thừa ký tự cuối
PASS  'abba'/'dogcatcatdog'
PASS  'abba'/'dogdogdogdog'
PASS  1 ký tự nuốt cả s dài 20
PASS  10 ký tự khác nhau, s dài 20
PASS  khó: abab/redblueredblue
PASS  khó: aaaa/asdasdasdasd
PASS  khó: aabb/xyzabcxzyabc
PASS  khó: abba/dogcatcatdog
PASS  khó: abba/dogdogdogdog
PASS  khó: abababab/xyxyxyxyxyxyxyxyxyyx


PASS  khó: abcdabcdab/xyzwxyzwxyzwxyzwxyzw


PASS  khó: abcabcabcd/xxxxxxxxxxxxxxxxxxxx


PASS  khó: ababababab/xxxxxxxxxxxxxxxxxxxy


PASS  khó: abcdefg/xxxxxxxxxxxxxxxxxxxx


## 12. Test đối chiếu chéo trên input ngẫu nhiên

Ta sinh nhiều input nhỏ với `random.seed` cố định (ai chạy lại cũng ra cùng kết quả):

- `pattern` dài 1..6, chỉ dùng chữ `a`, `b`, `c` (ít ký tự khác nhau → hay có ký tự lặp, dễ có đáp án `True`).
- `s` dài 1..10, chỉ dùng chữ `x`, `y` (bảng chữ nhỏ → nhiều chuỗi con giống nhau → hay "đụng" điều kiện song ánh).

Brute force (Cách 1) là **trọng tài**: Cách 2, 3, 4 và `Solution` phải cho cùng kết quả trên mọi input. Ta cũng đếm số ca `True` để chắc bộ test **không** toàn `False` (một bộ test toàn `False` sẽ không bắt được lỗi "trả `False` sai").

Phần cuối của cell thêm 200 input ngẫu nhiên dài **đúng 20** (biên constraints) cùng các input khó ở mục 9, để chắc `Solution` chạy nhanh ở trường hợp lớn nhất; 50 ca cuối được đối chiếu lại với brute force.

In [29]:
import random

random.seed(291)
methods = {
    "backtrack": word_pattern_match_backtrack,
    "pruned": word_pattern_match_pruned,
    "lengths": word_pattern_match_lengths,
    "Solution": lambda p, s: Solution().wordPatternMatch(p, s),
}

num_tests, num_true = 3000, 0
for _ in range(num_tests):
    p = "".join(random.choice("abc") for _ in range(random.randint(1, 6)))
    s = "".join(random.choice("xy") for _ in range(random.randint(1, 10)))
    expected = word_pattern_match_bruteforce(p, s)
    for name, fn in methods.items():
        got = fn(p, s)
        assert got == expected, (name, p, s, got, expected)
    num_true += expected

print(f"{num_tests} test ngẫu nhiên: 4 cách khớp brute force | số ca True: {num_true}, False: {num_tests - num_true}")

import time

# Thêm vài input ngẫu nhiên "to" (n = 20) để chắc Solution chạy nhanh ở biên constraints
random.seed(2910)
big_inputs = list(hard_cases)
for _ in range(200):
    p = "".join(random.choice("abcd") for _ in range(random.randint(2, 10)))
    s = "".join(random.choice("xy") for _ in range(20))
    big_inputs.append((p, s))

start = time.perf_counter()
answers = [Solution().wordPatternMatch(p, s) for p, s in big_inputs]
elapsed = time.perf_counter() - start

for (p, s), ans in zip(big_inputs[-50:], answers[-50:]):      # đối chiếu 50 ca cuối với brute force
    assert ans == word_pattern_match_bruteforce(p, s), (p, s)
print(f"{len(big_inputs)} input với n = 20: số True = {sum(answers)} | chạy xong dưới 1 giây? {elapsed < 1.0}")

3000 test ngẫu nhiên: 4 cách khớp brute force | số ca True: 1269, False: 1731


210 input với n = 20: số True = 86 | chạy xong dưới 1 giây? True


## 13. Vì sao thuật toán đúng? (Cách 2 / `Solution`)

**Invariant (bất biến)** — mỗi khi `backtrack(i, j)` được gọi:

1. `mapping` chứa **đúng** các ký tự xuất hiện trong `pattern[:i]`;
2. nối `mapping[c]` với `c` chạy qua `pattern[:i]` cho ra **đúng** `s[:j]`;
3. các chuỗi trong `mapping` **khác nhau từng đôi**, và `used == set(mapping.values())`.

Ban đầu (`i = j = 0`, `mapping` rỗng) cả 3 điều hiển nhiên đúng. Nhánh "đã có ánh xạ" chỉ đi tiếp khi `s` tại `j` bắt đầu bằng đúng `mapping[ch]` → điều 2 vẫn đúng ở `(i + 1, j + len)`. Nhánh "ký tự mới" chỉ gán `cand` chưa có trong `used` → điều 3 vẫn đúng; và mọi thay đổi đều được **gỡ** trước khi thử ứng viên khác → khi quay về, trạng thái y như trước.

**Không trả `True` sai (soundness):** chỉ trả `True` khi `i == m` và `j == n`. Theo invariant, khi đó nối các chuỗi theo `pattern` ra đúng `s`, mọi chuỗi khác rỗng (vì `end > j`), và khác nhau từng đôi → đó là một song ánh hợp lệ.

**Không bỏ sót (completeness):** giả sử tồn tại ánh xạ hợp lệ `M*`. Đi theo `M*` từ gốc: ở ký tự cũ, `mapping[ch] = M*[ch]` nên `startswith` đúng; ở ký tự mới, vòng lặp có thử ứng viên đúng bằng `M*[ch]` (độ dài của nó `<= max_len` theo lập luận ở mục 9), và ứng viên này **không** nằm trong `used` (vì `used` chỉ chứa chuỗi của các ký tự khác, mà `M*` là đơn ánh). Vậy thuật toán đi được hết đường của `M*` tới `i == m`, `j == n` — trừ khi nó đã tìm ra một ánh xạ hợp lệ khác và trả `True` sớm hơn. Cả hai trường hợp đều trả `True`.

**Vì sao không dùng memo theo `(i, j)` như Word Break?** Vì kết quả của `backtrack(i, j)` còn phụ thuộc vào `mapping` hiện tại — cùng `(i, j)` nhưng `mapping` khác thì đáp án có thể khác. Memo chỉ theo `(i, j)` sẽ cho đáp án sai (xem Lỗi 6 ở mục 15).

## 14. Độ phức tạp và so sánh các cách

Ký hiệu: `m = len(pattern)`, `n = len(s)`, `k` = số ký tự khác nhau trong `pattern` (`k <= m`, `k <= 26`).

| Cách | Ý tưởng | Time (trường hợp xấu, cận trên thô) | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force cắt | `combinations` chọn `m - 1` vị trí cắt, kiểm tra song ánh bằng `dict` + `set` | `O(C(n-1, m-1) · (n + m))` | `O(n + m)` | Trọng tài khi test; dễ viết, khó sai |
| 2. Backtracking | Gán dần; ký tự cũ chỉ kiểm `startswith`, ký tự mới thử mọi tiền tố chưa dùng | `O(n^k · m · n)` | `O(m + n)` | **Cách chính**, dễ giải thích khi phỏng vấn |
| 3. Backtracking + cắt tỉa | Thêm `min_need` và chặn trên `max_len` cho ký tự mới | Như Cách 2 về cận trên, nhưng ít lời gọi hơn nhiều khi `pattern` có ký tự lặp | `O(m + n)` | **Nộp bài** (bản `Solution`) |
| 4. Liệt kê bộ độ dài | Giải `Σ count·len = n` rồi dựng & kiểm tra | `O(C(n-1, k-1) · (m + n))` | `O(k + n)` | Khi `k` nhỏ; góc nhìn "đại số" thú vị |

Vài lưu ý quan trọng:

- **Mọi cách đều hàm mũ trong trường hợp xấu nhất.** Đây không phải do ta chưa đủ giỏi: bài toán tổng quát "có tồn tại phép thay mỗi ký tự bằng một chuỗi khác rỗng để `pattern` biến thành `s` không?" (không đòi song ánh) đã được chứng minh là **NP-complete** (Ehrenfeucht & Rozenberg, 1979), và với điều kiện song ánh cũng chưa ai biết thuật toán đa thức. Đó là lý do constraints chỉ cho `n <= 20`.
- Các cận trên trong bảng rất "thô". Con số thực tế (đếm ở mục 8 và 9) nhỏ hơn rất nhiều vì nhánh hỏng bị bỏ sớm.
- Space của cả 4 cách đều nhỏ; thứ "đắt" là thời gian.

## 15. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ giữ `dict` một chiều — quên `set used`

```python
for end in range(j + 1, n + 1):
    mapping[ch] = s[j:end]          # SAI: không kiểm tra chuỗi đã có chủ chưa
    if backtrack(i + 1, end): return True
    del mapping[ch]
```

Vì sao sai: hai ký tự khác nhau có thể nhận **cùng** một chuỗi → vi phạm song ánh. Phản ví dụ: `"ab"` / `"xx"` → trả `True` (với `a = b = "x"`), đúng ra `False`. Sửa: thêm `if cand in used: continue` và cập nhật `used` khi gán/gỡ.

### Lỗi 2: Quên **gỡ** (undo) khi quay lui

```python
mapping[ch] = cand; used.add(cand)
if backtrack(i + 1, end): return True
# SAI: thiếu  del mapping[ch]; used.remove(cand)
```

Vì sao sai: ánh xạ của nhánh đã thất bại vẫn còn nằm lại, "đầu độc" các nhánh sau. Ví dụ `"abb"` / `"xxxx"`: đáp án đúng là `True` (`a = "xx"`, `b = "x"`), nhưng nhánh `a = "x"` thất bại mà để lại "rác": các ứng viên đã thử cho `b` (`"xx"`, `"xxx"`) vẫn nằm trong `used`, và `b` vẫn giữ ánh xạ cũ. Khi quay về thử `a = "xx"`, chuỗi `"xx"` bị coi là "đã có chủ" nên bị bỏ qua → nhánh đúng không bao giờ được thử → trả `False`. Chỉ quên **một** trong hai dòng gỡ (`del` hoặc `remove`) cũng đủ sai.

In [30]:
def bug_one_way(pattern: str, s: str) -> bool:
    m, n = len(pattern), len(s)
    mapping = {}

    def backtrack(i, j):
        if i == m:
            return j == n
        if j == n:
            return False
        ch = pattern[i]
        if ch in mapping:
            word = mapping[ch]
            return s.startswith(word, j) and backtrack(i + 1, j + len(word))
        for end in range(j + 1, n + 1):
            mapping[ch] = s[j:end]                # <-- SAI: không có set used
            if backtrack(i + 1, end):
                return True
            del mapping[ch]
        return False

    return backtrack(0, 0)


def bug_no_undo(pattern: str, s: str) -> bool:
    m, n = len(pattern), len(s)
    mapping, used = {}, set()

    def backtrack(i, j):
        if i == m:
            return j == n
        if j == n:
            return False
        ch = pattern[i]
        if ch in mapping:
            word = mapping[ch]
            return s.startswith(word, j) and backtrack(i + 1, j + len(word))
        for end in range(j + 1, n + 1):
            cand = s[j:end]
            if cand in used:
                continue
            mapping[ch] = cand
            used.add(cand)
            if backtrack(i + 1, end):
                return True
            # <-- SAI: quên gỡ mapping[ch] và used
        return False

    return backtrack(0, 0)


print("Lỗi 1 ('ab','xx')  :", bug_one_way("ab", "xx"), "| đúng:", Solution().wordPatternMatch("ab", "xx"))
print("Lỗi 2 ('abb','xxxx'):", bug_no_undo("abb", "xxxx"), "| đúng:", Solution().wordPatternMatch("abb", "xxxx"))

Lỗi 1 ('ab','xx')  : True | đúng: False
Lỗi 2 ('abb','xxxx'): False | đúng: True


### Lỗi 3: Base case chỉ kiểm `i == m` mà quên `j == n`

```python
if i == m:
    return True          # SAI: chưa chắc đã dùng hết s
```

Vì sao sai: đặt xong mọi ký tự của `pattern` nhưng `s` còn thừa đuôi thì **chưa** khớp. Phản ví dụ: `"aba"` / `"xyxz"`: `a = "x"`, `b = "y"` khớp `"xyx"` nhưng còn thừa `"z"` → bản lỗi trả `True`, đúng là `False`. Sửa: `return j == n`.

### Lỗi 4: `return` kết quả lời gọi con ngay trong vòng lặp

```python
for end in range(j + 1, n + 1):
    ...
    return backtrack(i + 1, end)     # SAI: chỉ thử đúng MỘT ứng viên
```

Vì sao sai: nếu ứng viên đầu tiên thất bại, hàm trả `False` luôn mà không thử ứng viên khác. Phản ví dụ: `"aa"` / `"xyxy"`: ứng viên đầu `a = "x"` hỏng, bản lỗi trả `False`; đúng là `True` (`a = "xy"`). Sửa: `if backtrack(...): return True`, và chỉ `return False` **sau** vòng lặp.

### Lỗi 5: Lệch 1 ở `range` — `range(j + 1, n)` thay vì `range(j + 1, n + 1)`

Vì sao sai: `end` không bao giờ bằng `n`, tức ký tự mới **không bao giờ được nhận phần đuôi** của `s`. Phản ví dụ nhỏ nhất: `"a"` / `"x"` → bản lỗi `False`, đúng `True`. Nhớ: `s[j:end]` không lấy `end`, nên muốn lấy tới hết chuỗi thì `end` phải đạt `n`, và `range` phải dừng ở `n + 1`.

In [31]:
def make_buggy(kind: str):
    # Tạo bản Cách 2 bị lỗi theo 'kind' để so sánh (chỉ khác đúng 1 chỗ)
    def solve(pattern: str, s: str) -> bool:
        m, n = len(pattern), len(s)
        mapping, used = {}, set()

        def backtrack(i, j):
            if i == m:
                return True if kind == "base_case" else j == n          # Lỗi 3
            if j == n:
                return False
            ch = pattern[i]
            if ch in mapping:
                word = mapping[ch]
                return s.startswith(word, j) and backtrack(i + 1, j + len(word))
            last = n if kind == "off_by_one" else n + 1                 # Lỗi 5
            for end in range(j + 1, last):
                cand = s[j:end]
                if cand in used:
                    continue
                mapping[ch] = cand
                used.add(cand)
                if kind == "return_early":                              # Lỗi 4
                    return backtrack(i + 1, end)
                if backtrack(i + 1, end):
                    return True
                del mapping[ch]
                used.remove(cand)
            return False

        return backtrack(0, 0)
    return solve


for kind, (p, s) in [("base_case", ("aba", "xyxz")), ("return_early", ("aa", "xyxy")), ("off_by_one", ("a", "x"))]:
    wrong, right = make_buggy(kind)(p, s), Solution().wordPatternMatch(p, s)
    print(f"{kind:13} ({p!r}, {s!r}): bản lỗi = {wrong!s:5} | đúng = {right}")
    assert wrong != right

base_case     ('aba', 'xyxz'): bản lỗi = True  | đúng = False
return_early  ('aa', 'xyxy'): bản lỗi = False | đúng = True
off_by_one    ('a', 'x'): bản lỗi = False | đúng = True


### Lỗi 6: Memo theo `(i, j)` như bài Word Break

Học xong Word Break (#139/#140), nhiều bạn thêm `@lru_cache` cho `backtrack(i, j)` để "tăng tốc":

```python
@lru_cache(maxsize=None)
def backtrack(i, j): ...        # SAI: kết quả còn phụ thuộc mapping!
```

Vì sao sai: memo chỉ đúng khi kết quả của hàm **chỉ phụ thuộc vào tham số**. Ở đây `backtrack(i, j)` còn phụ thuộc vào `mapping` và `used` (trạng thái chung). Lần đầu gặp `(i, j)` dưới một `mapping` nào đó mà thất bại, cache ghi `False`; lần sau gặp lại `(i, j)` với `mapping` **khác** (có thể thành công) thì cache vẫn trả `False`.

Phản ví dụ: `"abb"` / `"xxxx"`. Nhánh `a = "x"`, `b = "xx"` gọi `backtrack(2, 3)` và thất bại (cần `"xx"` ở vị trí 3 nhưng chỉ còn `"x"`) → cache `(2, 3) = False`. Sau đó nhánh đúng `a = "xx"`, `b = "x"` cũng gọi `backtrack(2, 3)` — lần này đáng lẽ thành công — nhưng nhận `False` từ cache.

### Lỗi 7: Ký tự đã có ánh xạ mà vẫn thử tiền tố mới

Nếu quên nhánh `if ch in mapping` và luôn chạy vòng thử ứng viên, ta sẽ **gán đè** ánh xạ cũ (`mapping[ch] = cand` ghi đè im lặng như ở mục 2.3) → một ký tự bị ánh xạ tới hai chuỗi khác nhau. Ví dụ `"aa"` / `"xy"` sẽ ra `True` sai. Sửa: ký tự cũ **chỉ kiểm tra**, không thử.

In [32]:
from functools import lru_cache


def bug_memo_ij(pattern: str, s: str) -> bool:
    m, n = len(pattern), len(s)
    mapping, used = {}, set()

    @lru_cache(maxsize=None)                       # <-- SAI: bỏ qua mapping
    def backtrack(i, j):
        if i == m:
            return j == n
        if j == n:
            return False
        ch = pattern[i]
        if ch in mapping:
            word = mapping[ch]
            return s.startswith(word, j) and backtrack(i + 1, j + len(word))
        for end in range(j + 1, n + 1):
            cand = s[j:end]
            if cand in used:
                continue
            mapping[ch] = cand
            used.add(cand)
            if backtrack(i + 1, end):
                return True
            del mapping[ch]
            used.remove(cand)
        return False

    return backtrack(0, 0)


print("Lỗi 6 ('abb','xxxx'):", bug_memo_ij("abb", "xxxx"), "| đúng:", Solution().wordPatternMatch("abb", "xxxx"))
print("Kiểm tra ánh xạ đúng:", is_valid_mapping("abb", "xxxx", {"a": "xx", "b": "x"}))

Lỗi 6 ('abb','xxxx'): False | đúng: True
Kiểm tra ánh xạ đúng: True


## 16. Mẹo nhận dạng pattern

- **"Có tồn tại một cách gán / ánh xạ thỏa mọi ràng buộc không?"** + kích thước rất nhỏ (`<= 20`) → nghĩ ngay tới **backtracking**: gán dần, kiểm tra sớm, sai thì lùi.
- **"Song ánh", "một-một", "không hai cái nào trùng"** → `dict` cho chiều xuôi + `set` (hoặc `dict` thứ hai) cho chiều ngược. Luôn tự hỏi: *"mình đã kiểm tra cả hai chiều chưa?"* và thử ngay phản ví dụ kiểu `"ab"` / `"xx"`.
- **"Cắt chuỗi thành các mảnh mà không biết ranh giới"** → vòng `for end in range(j + 1, n + 1)` với mảnh `s[j:end]` (khung giống Palindrome Partitioning, Word Break).
- **"Khi đã chọn thì các lần sau bị ép"** (ký tự đã gán) → nhánh **kiểm tra**, không phải nhánh **thử** — đây là nguồn tiết kiệm lớn nhất của backtracking.
- **"Tài nguyên còn lại có giới hạn"** (số ký tự còn lại, tổng còn lại, số ô còn trống) → **cắt tỉa**: ước lượng nhu cầu tối thiểu của phần còn lại, nếu vượt quá tài nguyên thì lùi ngay.
- **Kết quả phụ thuộc vào "lịch sử lựa chọn"** (ở đây là `mapping`) → **không** memo đơn giản theo vị trí được. Ngược lại, nếu bài chỉ phụ thuộc vị trí (Word Break) thì memo/DP.

## 17. Biến thể và bài liên quan

| Bài | Liên hệ với #291 |
|---|---|
| **290. Word Pattern** (Easy) | Bản có dấu cách: cách cắt đã biết, chỉ còn kiểm tra song ánh bằng `dict` + `set` — `O(m + n)`. |
| **205. Isomorphic Strings** (Easy) | Song ánh giữa **ký tự ↔ ký tự** của hai chuỗi cùng độ dài; đúng kỹ thuật hai chiều ở mục 3. |
| **890. Find and Replace Pattern** (Medium) | Kiểm tra song ánh ký tự ↔ ký tự cho **nhiều** từ với cùng một `pattern`. |
| **139/140. Word Break I/II** | Cũng "cắt chuỗi không biết ranh giới", nhưng **từ điển cố định** → kết quả chỉ phụ thuộc vị trí → memo/DP được. Ở #291 "từ điển" (`mapping`) được **tự xây** trong lúc chạy → phải backtracking. |
| **131. Palindrome Partitioning** (Medium) | Cùng khung backtracking `for end in range(start + 1, n + 1)` để cắt chuỗi. |
| **10. Regular Expression Matching / 44. Wildcard Matching** (Hard) | Cũng "khớp mẫu", nhưng `.`, `*`, `?` **không nhớ** đã khớp gì → DP theo `(i, j)` đúng. Ở #291 mỗi ký tự **nhớ** chuỗi của nó → không DP được. |
| Biến thể tự chế | Bỏ điều kiện song ánh (Bài 5), đếm số ánh xạ hợp lệ (Bài 6), trả về chính ánh xạ tìm được, cho `pattern` chứa cả ký tự "hằng" phải khớp đúng nguyên văn. |

## 18. Bài tập vận dụng

### Bài 1 — Đoán output (không chạy code)

`Solution().wordPatternMatch(pattern, s)` trả về gì với từng cặp sau? Nếu `True`, ghi ra **một** ánh xạ hợp lệ.

```python
("ab",   "aa")
("aaa",  "xxx")
("aba",  "xyzx")
("abc",  "xx")
("aab",  "xxx")
("abab", "xyzxyz")
```

### Bài 2 — Tính tay bằng phương trình độ dài

a) Ví dụ 3: `pattern = "aabb"`, `s = "xyzabcxzyabc"` (`n = 12`). Viết phương trình độ dài, suy ra `len_a` chỉ có thể là `1..5`. Lập bảng với mỗi `len_a`: so `s[0:len_a]` với `s[len_a:2·len_a]`. Từ đó giải thích vì sao đáp án là `False`.

b) `pattern = "abba"`, `s = "dogcatcatdog"`. Viết phương trình, liệt kê các cặp `(len_a, len_b)`, kiểm tra từng cặp (gợi ý: `a` phải vừa là **tiền tố** vừa là **hậu tố** của `s`). Có bao nhiêu ánh xạ hợp lệ?

### Bài 3 — Điền TODO: Word Pattern I (#290) bằng `dict` + `set`

Hoàn thiện hàm kiểm tra song ánh **hai chiều** khi `s` có dấu cách.

In [33]:
def exercise_word_pattern(pattern: str, s: str) -> bool:
    words = s.split()
    # TODO 1: nếu số từ khác số ký tự pattern -> False
    char_to_word = {}
    used = set()
    # TODO 2: duyệt từng cặp (ch, w) bằng zip
    # TODO 3: ch đã có ánh xạ -> phải bằng w
    # TODO 4: ch mới -> w không được nằm trong used; rồi lưu ch -> w và thêm w vào used
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_word_pattern("abba", "dog cat cat dog") == True
# assert exercise_word_pattern("abba", "dog cat cat fish") == False
# assert exercise_word_pattern("aaaa", "dog cat cat dog") == False
# assert exercise_word_pattern("abba", "dog dog dog dog") == False

### Bài 4 — Tự viết lại Cách 2 (backtracking)

Không nhìn lên trên, tự viết `exercise_word_pattern_match(pattern, s)`. Nhớ 4 điểm: base case `i == m` trả `j == n`; ký tự cũ chỉ **kiểm** `startswith`; ký tự mới **thử** `s[j:end]` với `end` chạy tới `n` (bao gồm `n`) và bỏ qua chuỗi trong `used`; gán thì **gỡ**.

In [34]:
def exercise_word_pattern_match(pattern: str, s: str) -> bool:
    # TODO: mapping (dict) + used (set) + hàm lồng backtrack(i, j)
    raise NotImplementedError

# Gợi ý test:
# assert exercise_word_pattern_match("abab", "redblueredblue") == True
# assert exercise_word_pattern_match("aaaa", "asdasdasdasd") == True
# assert exercise_word_pattern_match("aabb", "xyzabcxzyabc") == False
# assert exercise_word_pattern_match("ab", "xx") == False
# assert exercise_word_pattern_match("aba", "xyxz") == False

### Bài 5 — Biến thể: bỏ điều kiện song ánh

Giả sử đề đổi: chỉ cần mỗi ký tự ánh xạ tới **một** chuỗi khác rỗng (chiều 1), còn **hai ký tự khác nhau được phép** dùng chung một chuỗi. Viết `exercise_match_not_injective(pattern, s)`.

Ví dụ: `"ab"` / `"xx"` → `True` (`a = b = "x"`); `"abba"` / `"dogdogdogdog"` → `True`; `"aabb"` / `"xyzabcxzyabc"` → vẫn `False`.

Câu hỏi thêm: nếu bản song ánh trả `True` thì bản này chắc chắn trả gì? Điều ngược lại có đúng không?

In [35]:
def exercise_match_not_injective(pattern: str, s: str) -> bool:
    # TODO: giống Cách 2 nhưng bỏ hẳn phần liên quan tới set used
    pass

# Gợi ý test:
# assert exercise_match_not_injective("ab", "xx") == True
# assert exercise_match_not_injective("abba", "dogdogdogdog") == True
# assert exercise_match_not_injective("aabb", "xyzabcxzyabc") == False
# assert exercise_match_not_injective("abc", "xx") == False

### Bài 6 — Mini challenge: đếm **tất cả** ánh xạ hợp lệ

Viết `exercise_count_mappings(pattern, s)` trả về **số** ánh xạ song ánh hợp lệ (không dừng ở ánh xạ đầu tiên!).

Ví dụ: `"abab"` / `"redblueredblue"` → `6`; `"aaaa"` / `"asdasdasdasd"` → `1`; `"aabb"` / `"xyzabcxzyabc"` → `0`; `"ab"` / `"xyz"` → `2`; `"abba"` / `"dogcatcatdog"` → `1`.

Yêu cầu: dùng backtracking (có thể kèm cắt tỉa). Thử tự kiểm tra kết quả bằng brute force: mỗi ánh xạ hợp lệ ứng với **đúng một** cách cắt hợp lệ, nên có thể đếm bằng `combinations`.

In [36]:
def exercise_count_mappings(pattern: str, s: str) -> int:
    # TODO: hàm lồng count(i, j) trả về SỐ cách hoàn thành từ trạng thái (i, j)
    pass

# Gợi ý test:
# assert exercise_count_mappings("abab", "redblueredblue") == 6
# assert exercise_count_mappings("aaaa", "asdasdasdasd") == 1
# assert exercise_count_mappings("aabb", "xyzabcxzyabc") == 0
# assert exercise_count_mappings("ab", "xyz") == 2
# assert exercise_count_mappings("abba", "dogcatcatdog") == 1

## 19. Gợi ý theo mức

**Bài 1**
- Mức 1: kiểm tra 4 điều ở mục 1 (đặc biệt điều 3 "khác nhau" và điều 4 "dùng hết `s`").
- Mức 2: với `pattern` có ký tự lặp, viết phương trình độ dài trước (`"aab"`: `2·len_a + len_b = 3` → chỉ có `len_a = len_b = 1`).
- Mức 3: `"abc"` / `"xx"`: `m > n` thì sao?

**Bài 2**
- Mức 1: `2·len_a + 2·len_b = 12` → `len_a + len_b = 6`, mà `len_b >= 1`.
- Mức 2: hai mảnh đầu đều là `a` → phải bằng nhau: `s[0:L] == s[L:2L]`.
- Mức 3: câu b: `a` là tiền tố và hậu tố cùng độ dài → so `s[:L]` với `s[-L:]`.

**Bài 3**
- Mức 1: mỗi cặp `(ch, w)` rơi vào đúng một trong hai nhánh: `ch` cũ hoặc `ch` mới.
- Mức 2: nhánh cũ kiểm `char_to_word[ch] != w`; nhánh mới kiểm `w in used`.
- Mức 3: đừng quên `return True` ở cuối, và kiểm tra độ dài trước khi `zip`.

**Bài 4**
- Mức 1: trạng thái gồm `(i, j)` + `mapping` + `used`; hàm trả `bool`.
- Mức 2: khung `chọn → đi tiếp → gỡ` y như `split_into_k` ở mục 7.
- Mức 3: `if backtrack(i + 1, end): return True` — **không** `return backtrack(...)` trong vòng lặp.

**Bài 5**
- Mức 1: chiều 2 của song ánh nằm ở đâu trong code Cách 2? Xóa nó.
- Mức 2: không cần `used` nữa; nhánh ký tự mới thử mọi `s[j:end]`.
- Mức 3: song ánh `True` ⇒ bản này `True` (vì ít ràng buộc hơn); ngược lại không đúng (`"ab"` / `"xx"`).

**Bài 6**
- Mức 1: thay vì trả `bool`, hàm con trả **số** cách hoàn thành.
- Mức 2: base case: `1` nếu `j == n` else `0`; ký tự cũ: trả số của lời gọi con (hoặc `0`); ký tự mới: **cộng dồn** qua mọi ứng viên.
- Mức 3: vẫn phải gỡ sau **mỗi** ứng viên, vì ta không dừng sớm.

## 20. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

**Bài 1:**

| Cặp | Kết quả | Giải thích |
|---|---|---|
| `"ab"`, `"aa"` | `False` | `a = "a"` thì `b` chỉ còn `"a"` (đã có chủ); `a = "aa"` thì `b` không còn gì |
| `"aaa"`, `"xxx"` | `True` | `a = "x"` |
| `"aba"`, `"xyzx"` | `True` | `a = "x"`, `b = "yz"` |
| `"abc"`, `"xx"` | `False` | 3 ký tự cần ít nhất 3 ký tự của `s` |
| `"aab"`, `"xxx"` | `False` | `2·len_a + len_b = 3` → `a = "x"`, `b = "x"` trùng nhau → vi phạm song ánh |
| `"abab"`, `"xyzxyz"` | `True` | `len_a + len_b = 3`: `a = "x"`, `b = "yz"` (hoặc `a = "xy"`, `b = "z"`) |

**Bài 2a:** `2·len_a + 2·len_b = 12`, `len_b >= 1` → `len_a <= 5`.

| `len_a` | `s[0:len_a]` | `s[len_a:2·len_a]` | Bằng nhau? |
|---:|---|---|---|
| 1 | `x` | `y` | ✗ |
| 2 | `xy` | `za` | ✗ |
| 3 | `xyz` | `abc` | ✗ |
| 4 | `xyza` | `bcxz` | ✗ |
| 5 | `xyzab` | `cxzya` | ✗ |

Không `len_a` nào làm hai mảnh `a` bằng nhau → `False`. (Đây đúng là 5 nhánh trong trace Cách 3.)

**Bài 2b:** `2·len_a + 2·len_b = 12` → `len_a + len_b = 6` → `(1,5), (2,4), (3,3), (4,2), (5,1)`. Chỉ `len_a = 3` cho tiền tố `"dog"` = hậu tố `"dog"`; khi đó `b = s[3:6] = "cat"`, lần hai `s[6:9] = "cat"` ✓, `"dog" != "cat"` ✓. Vậy có đúng **1** ánh xạ hợp lệ.

Cell dưới kiểm chứng Bài 1 và Bài 2 bằng code.

In [37]:
sol = Solution()
bai1 = [("ab", "aa", False), ("aaa", "xxx", True), ("aba", "xyzx", True),
        ("abc", "xx", False), ("aab", "xxx", False), ("abab", "xyzxyz", True)]
for p, s, expected in bai1:
    assert sol.wordPatternMatch(p, s) == expected, (p, s)
print("Bài 1: PASS")

s3 = "xyzabcxzyabc"
for L in range(1, 6):
    print(f"  len_a={L}: {s3[0:L]!r:8} vs {s3[L:2 * L]!r:8} -> {s3[0:L] == s3[L:2 * L]}")

s2b = "dogcatcatdog"
valid = []
for La in range(1, 6):
    Lb = 6 - La
    mp = build_mapping("abba", s2b, {"a": La, "b": Lb})
    if mp is not None and is_valid_mapping("abba", s2b, mp):
        valid.append(mp)
print("Bài 2b: các ánh xạ hợp lệ =", valid)
assert valid == [{"a": "dog", "b": "cat"}]

Bài 1: PASS
  len_a=1: 'x'      vs 'y'      -> False
  len_a=2: 'xy'     vs 'za'     -> False
  len_a=3: 'xyz'    vs 'abc'    -> False
  len_a=4: 'xyza'   vs 'bcxz'   -> False
  len_a=5: 'xyzab'  vs 'cxzya'  -> False
Bài 2b: các ánh xạ hợp lệ = [{'a': 'dog', 'b': 'cat'}]


In [38]:
def answer_word_pattern(pattern: str, s: str) -> bool:
    words = s.split()
    if len(words) != len(pattern):
        return False
    char_to_word, used = {}, set()
    for ch, w in zip(pattern, words):
        if ch in char_to_word:
            if char_to_word[ch] != w:
                return False
        else:
            if w in used:
                return False
            char_to_word[ch] = w
            used.add(w)
    return True


def answer_word_pattern_match(pattern: str, s: str) -> bool:
    m, n = len(pattern), len(s)
    mapping, used = {}, set()

    def backtrack(i, j):
        if i == m:
            return j == n
        if j == n:
            return False
        ch = pattern[i]
        if ch in mapping:
            word = mapping[ch]
            return s.startswith(word, j) and backtrack(i + 1, j + len(word))
        for end in range(j + 1, n + 1):
            cand = s[j:end]
            if cand in used:
                continue
            mapping[ch] = cand
            used.add(cand)
            if backtrack(i + 1, end):
                return True
            del mapping[ch]
            used.remove(cand)
        return False

    return backtrack(0, 0)


def answer_match_not_injective(pattern: str, s: str) -> bool:
    m, n = len(pattern), len(s)
    mapping = {}

    def backtrack(i, j):
        if i == m:
            return j == n
        if j == n:
            return False
        ch = pattern[i]
        if ch in mapping:
            word = mapping[ch]
            return s.startswith(word, j) and backtrack(i + 1, j + len(word))
        for end in range(j + 1, n + 1):
            mapping[ch] = s[j:end]
            if backtrack(i + 1, end):
                return True
            del mapping[ch]
        return False

    return backtrack(0, 0)


def answer_count_mappings(pattern: str, s: str) -> int:
    m, n = len(pattern), len(s)
    mapping, used = {}, set()

    def count(i, j):
        if i == m:
            return 1 if j == n else 0
        if j == n:
            return 0
        ch = pattern[i]
        if ch in mapping:
            word = mapping[ch]
            return count(i + 1, j + len(word)) if s.startswith(word, j) else 0
        total = 0
        for end in range(j + 1, n + 1):
            cand = s[j:end]
            if cand in used:
                continue
            mapping[ch] = cand
            used.add(cand)
            total += count(i + 1, end)       # cộng dồn, KHÔNG dừng sớm
            del mapping[ch]
            used.remove(cand)
        return total

    return count(0, 0)


# Bài 3
assert answer_word_pattern("abba", "dog cat cat dog") == True
assert answer_word_pattern("abba", "dog cat cat fish") == False
assert answer_word_pattern("aaaa", "dog cat cat dog") == False
assert answer_word_pattern("abba", "dog dog dog dog") == False
# Bài 4
assert answer_word_pattern_match("abab", "redblueredblue") == True
assert answer_word_pattern_match("aaaa", "asdasdasdasd") == True
assert answer_word_pattern_match("aabb", "xyzabcxzyabc") == False
assert answer_word_pattern_match("ab", "xx") == False
assert answer_word_pattern_match("aba", "xyxz") == False
# Bài 5
assert answer_match_not_injective("ab", "xx") == True
assert answer_match_not_injective("abba", "dogdogdogdog") == True
assert answer_match_not_injective("aabb", "xyzabcxzyabc") == False
assert answer_match_not_injective("abc", "xx") == False
# Bài 6
assert answer_count_mappings("abab", "redblueredblue") == 6
assert answer_count_mappings("aaaa", "asdasdasdasd") == 1
assert answer_count_mappings("aabb", "xyzabcxzyabc") == 0
assert answer_count_mappings("ab", "xyz") == 2
assert answer_count_mappings("abba", "dogcatcatdog") == 1

# Đối chiếu ngẫu nhiên: Bài 4 với Solution; Bài 5 "yếu hơn" song ánh; Bài 6 với đếm bằng combinations
random.seed(6)
for _ in range(1500):
    p = "".join(random.choice("abc") for _ in range(random.randint(1, 5)))
    s = "".join(random.choice("xy") for _ in range(random.randint(1, 9)))
    bij = Solution().wordPatternMatch(p, s)
    assert answer_word_pattern_match(p, s) == bij
    if bij:
        assert answer_match_not_injective(p, s)
    brute_count = sum(matches_pieces(p, cut_by_positions(s, c))
                      for c in combinations(range(1, len(s)), len(p) - 1)) if len(p) <= len(s) else 0
    assert answer_count_mappings(p, s) == brute_count
    assert (brute_count > 0) == bij

print("Đáp án Bài 3–6: PASS toàn bộ test (kể cả 1500 test ngẫu nhiên)")

Đáp án Bài 3–6: PASS toàn bộ test (kể cả 1500 test ngẫu nhiên)


## 21. Template cuối cùng nên nhớ

```python
class Solution:
    def wordPatternMatch(self, pattern: str, s: str) -> bool:
        m, n = len(pattern), len(s)
        mapping, used = {}, set()              # chiều xuôi + chiều ngược của song ánh

        def backtrack(i, j):
            if i == m:
                return j == n                  # hết pattern thì phải hết s
            if n - j < m - i:                  # cắt tỉa tối thiểu
                return False
            ch = pattern[i]
            if ch in mapping:                  # đã biết -> chỉ KIỂM
                word = mapping[ch]
                return s.startswith(word, j) and backtrack(i + 1, j + len(word))
            for end in range(j + 1, n + 1):    # chưa biết -> THỬ (end tới n!)
                cand = s[j:end]
                if cand in used:
                    continue
                mapping[ch] = cand; used.add(cand)        # chọn
                if backtrack(i + 1, end):                 # đi tiếp
                    return True
                del mapping[ch]; used.remove(cand)        # gỡ
            return False

        return backtrack(0, 0)
```

Câu thần chú: **"Đã biết thì kiểm, chưa biết thì thử; thử xong phải gỡ; song ánh giữ hai chiều (`dict` + `set`)."**

Muốn nhanh hơn: thay `n - j < m - i` bằng `min_need` (tính cả độ dài đã biết) và chặn `len <= (n - j - others) // cnt` cho ký tự mới — như bản `Solution` ở mục 11.

## 22. Checklist tự kiểm tra trước khi submit

- Đã kiểm tra **cả hai chiều** của song ánh chưa? (`dict mapping` + `set used`; thử ngay `"ab"` / `"xx"` phải ra `False`.)
- Base case có phải `return j == n` (dùng **hết** `s`) chứ không phải `return True`?
- Ký tự **đã có** ánh xạ có đi nhánh **kiểm tra** `startswith` (không thử, không gán đè) chưa?
- Vòng thử ứng viên có chạy `end` tới **`n`** (`range(j + 1, n + 1)`) chưa? Mỗi mảnh có **khác rỗng** (`end > j`) chưa?
- Có **gỡ** cả `mapping` lẫn `used` sau mỗi ứng viên thất bại chưa?
- Có viết `if backtrack(...): return True` (không phải `return backtrack(...)` trong vòng lặp) chưa?
- Có lỡ thêm memo theo `(i, j)` không? (Sai, vì kết quả phụ thuộc `mapping`.)
- `mapping`/`used` có được tạo **mới** cho mỗi lần gọi `wordPatternMatch` chưa?
- Đã test: 3 ví dụ đề, `pattern` dài hơn `s`, 1 ký tự, `"ab"`/`"aa"`, `"aba"`/`"xyxz"` (thừa đuôi), `"aa"`/`"xyx"` (độ dài lẻ), và đối chiếu với brute force trên input ngẫu nhiên?